# Les mathématiques de *Attention Is All You Need* — en partant de zéro

**Basé sur** : Vaswani et al., *Attention Is All You Need*, NIPS 2017 (arXiv:1706.03762)

---

L'idée de ce notebook c'est de reprendre le papier *Attention Is All You Need* et d'expliquer **toutes** les maths qui sont derrière, en partant vraiment de zéro. Chaque concept est défini ou prouvé avant d'être utilisé. Rien ne tombe du ciel.

Le but : qu'un mec qui connait les bases des maths (algèbre, fonctions, sommations) puisse suivre de A à Z et comprendre pourquoi chaque choix a été fait.

**Plan** :
- **Partie 0** — Outils mathématiques de base (vecteurs, produit scalaire, matrices, probas, softmax)
- **Parcours de l'article** — Résumé section par section du papier
- **Partie 1** — Réseau de neurones : le minimum vital
- **Partie 2** — Modéliser des séquences (RNN, convolutions, et pourquoi on veut autre chose)
- **Partie 3** — L'attention : traitement mathématique complet
- **Partie 4** — Multi-Head Attention
- **Partie 5** — Architecture complète du Transformer
- **Partie 6** — Analyse de complexité détaillée
- **Partie 7** — Exemple complet : trace d'un encoder layer
- **Partie 8** — Implémentation from scratch
- **Partie 9** — Exercices conceptuels
- **Partie 10** — Test final : "les yeux fermés"
- **Partie 11** — Pièges courants d'implémentation

In [ ]:
# Installation des dépendances
# Exécuter cette cellule une seule fois pour créer le venv et installer torch
import subprocess, sys, os

VENV_DIR = ".venv"

if not os.path.exists(VENV_DIR):
    print("Création du venv...")
    subprocess.check_call([sys.executable, "-m", "venv", VENV_DIR])
    print("venv créé.")

# Installer les dépendances dans le venv
pip = os.path.join(VENV_DIR, "bin", "pip")
if not os.path.exists(pip):
    pip = os.path.join(VENV_DIR, "Scripts", "pip")  # Windows

deps = ["torch", "numpy"]
print("Installation des dépendances...")
subprocess.check_call([pip, "install", "-q"] + deps)
print("Tout est installé. Si le kernel n'est pas le bon, sélectionner .venv comme kernel.")


## Notations utilisées dans tout le notebook

| Symbole | Signification | Valeur (base model) |
|---|---|---|
| $B$ | taille du batch | variable |
| $T$ (ou $n$, $m$) | longueur de séquence | variable |
| $d_{\text{model}}$ | dimension cachée | 512 |
| $h$ | nombre de têtes d'attention | 8 |
| $d_k = d_v$ | dimension par tête ($d_{\text{model}} / h$) | 64 |
| $d_{ff}$ | dimension interne du FFN | 2048 |
| $N$ | nombre de couches encoder / decoder | 6 |
| $V$ | taille du vocabulaire | 37000 |
| $PE$ | positional encoding | sinusoïdal |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
torch.set_printoptions(precision=4, linewidth=120)

/Users/vitt/school/attention-is-all-you-need/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


---
# Partie 0 — Outils mathématiques de base

Avant de toucher au Transformer, il faut poser les briques. Tout ce qui suit sera utilisé plus tard.

## 0.1 — Vecteurs

### Définition

Un **vecteur** dans $\mathbb{R}^n$, c'est une liste ordonnée de $n$ nombres réels :

$$\mathbf{v} = (v_1, v_2, \ldots, v_n) \in \mathbb{R}^n$$

Par exemple, $\mathbf{v} = (3, -1, 4)$ est un vecteur dans $\mathbb{R}^3$.

### Opérations

**Addition** : on additionne composante par composante.

$$\mathbf{a} + \mathbf{b} = (a_1 + b_1, \, a_2 + b_2, \, \ldots, \, a_n + b_n)$$

**Multiplication par un scalaire** $\lambda \in \mathbb{R}$ :

$$\lambda \mathbf{a} = (\lambda a_1, \, \lambda a_2, \, \ldots, \, \lambda a_n)$$

### Interprétation géométrique

En 2D ou 3D, un vecteur c'est une flèche : une direction et une longueur. En dimension 512 (ce qu'utilise le Transformer), on ne peut plus dessiner, mais l'intuition reste la même.

### Norme (longueur)

La **norme euclidienne** d'un vecteur $\mathbf{v}$, notée $\|\mathbf{v}\|$, c'est sa "longueur" :

$$\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2} = \sqrt{\sum_{i=1}^{n} v_i^2}$$

Exemple : $\mathbf{v} = (3, 4)$, alors $\|\mathbf{v}\| = \sqrt{9 + 16} = \sqrt{25} = 5$.

Un **vecteur unitaire** est un vecteur de norme 1. On peut normaliser n'importe quel vecteur non nul : $\hat{\mathbf{v}} = \mathbf{v} / \|\mathbf{v}\|$.

In [2]:
# Vérification des opérations sur les vecteurs
a = torch.tensor([3.0, -1.0, 4.0])
b = torch.tensor([1.0, 2.0, -1.0])

print("a =", a)
print("b =", b)
print("a + b =", a + b)          # addition composante par composante
print("2 * a =", 2 * a)          # multiplication par un scalaire
print("||a|| =", torch.norm(a))  # norme euclidienne
print("||[3, 4]|| =", torch.norm(torch.tensor([3.0, 4.0])))  # = 5.0

# Vecteur unitaire
v = torch.tensor([3.0, 4.0])
v_hat = v / torch.norm(v)
print("v_hat =", v_hat, "  norme =", torch.norm(v_hat))  # norme = 1

a = tensor([ 3., -1.,  4.])
b = tensor([ 1.,  2., -1.])
a + b = tensor([4., 1., 3.])
2 * a = tensor([ 6., -2.,  8.])
||a|| = tensor(5.0990)
||[3, 4]|| = tensor(5.)
v_hat = tensor([0.6000, 0.8000])   norme = tensor(1.)


## 0.2 — Produit scalaire

### Définition algébrique

Le **produit scalaire** (dot product) de deux vecteurs $\mathbf{a}, \mathbf{b} \in \mathbb{R}^n$ est un nombre réel :

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i \, b_i = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n$$

Exemple : $\mathbf{a} = (1, 2, 3)$, $\mathbf{b} = (4, 5, 6)$.

$$\mathbf{a} \cdot \mathbf{b} = 1 \times 4 + 2 \times 5 + 3 \times 6 = 4 + 10 + 18 = 32$$

### Propriétés

- **Commutatif** : $\mathbf{a} \cdot \mathbf{b} = \mathbf{b} \cdot \mathbf{a}$ (changer l'ordre ne change rien)
- **Linéaire** : $\mathbf{a} \cdot (\lambda \mathbf{b} + \mathbf{c}) = \lambda (\mathbf{a} \cdot \mathbf{b}) + \mathbf{a} \cdot \mathbf{c}$
- **Lien avec la norme** : $\mathbf{a} \cdot \mathbf{a} = \|\mathbf{a}\|^2$

### Formule géométrique : preuve

On va montrer que $\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \, \|\mathbf{b}\| \cos\theta$ où $\theta$ est l'angle entre $\mathbf{a}$ et $\mathbf{b}$.

**Preuve.** On part de la loi des cosinus. Considérons le triangle formé par $\mathbf{a}$, $\mathbf{b}$, et $\mathbf{a} - \mathbf{b}$. La loi des cosinus donne :

$$\|\mathbf{a} - \mathbf{b}\|^2 = \|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - 2\|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$$

Développons le membre de gauche avec la définition de la norme :

$$\|\mathbf{a} - \mathbf{b}\|^2 = (\mathbf{a} - \mathbf{b}) \cdot (\mathbf{a} - \mathbf{b}) = \mathbf{a} \cdot \mathbf{a} - 2\,\mathbf{a} \cdot \mathbf{b} + \mathbf{b} \cdot \mathbf{b} = \|\mathbf{a}\|^2 - 2\,\mathbf{a} \cdot \mathbf{b} + \|\mathbf{b}\|^2$$

En comparant les deux expressions :

$$\|\mathbf{a}\|^2 - 2\,\mathbf{a} \cdot \mathbf{b} + \|\mathbf{b}\|^2 = \|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - 2\|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$$

On simplifie $\|\mathbf{a}\|^2 + \|\mathbf{b}\|^2$ des deux côtés :

$$-2\,\mathbf{a} \cdot \mathbf{b} = -2\|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$$

$$\boxed{\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta} \qquad \blacksquare$$

### Pourquoi ça mesure la similarité

$\cos\theta$ vaut :
- **1** quand $\theta = 0$ (même direction) → produit scalaire maximal
- **0** quand $\theta = 90°$ (perpendiculaires) → produit scalaire nul
- **$-1$** quand $\theta = 180°$ (directions opposées) → produit scalaire minimal

Donc : **plus deux vecteurs pointent dans la même direction, plus leur produit scalaire est grand**. C'est exactement ce qu'on va utiliser dans l'attention pour mesurer si une *query* ressemble à une *key*.

In [ ]:
# Produit scalaire
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print("a · b =", torch.dot(a, b))  # = 32

# Vérification de la formule géométrique en 2D
a2 = torch.tensor([1.0, 0.0])   # pointe vers la droite
b2 = torch.tensor([1.0, 1.0])   # pointe en diagonale (45°)

dot = torch.dot(a2, b2)
cos_theta = dot / (torch.norm(a2) * torch.norm(b2))
theta = torch.acos(cos_theta) * 180 / math.pi

print(f"a2 · b2 = {dot.item():.4f}")
print(f"cos(θ) = {cos_theta.item():.4f}")
print(f"θ = {theta.item():.1f}°")  # devrait être 45°

# Cas perpendiculaires : produit scalaire = 0
c = torch.tensor([1.0, 0.0])
d = torch.tensor([0.0, 1.0])
print(f"vecteurs perpendiculaires : c · d = {torch.dot(c, d).item()}")  # = 0

## 0.3 — Similarité cosinus

### Dérivation depuis le produit scalaire

On a montré que $\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$. En isolant $\cos\theta$ :

$$\cos\theta = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

C'est la **similarité cosinus**. Elle ne dépend que de la *direction* des vecteurs, pas de leur longueur.

### Pourquoi c'est utile

Imaginons deux vecteurs qui représentent des mots :
- $\mathbf{a}$ = vecteur de "chat" de norme 10
- $\mathbf{b}$ = vecteur de "chat" de norme 0.1 (même direction mais beaucoup plus court)
- $\mathbf{c}$ = vecteur de "voiture" qui pointe dans une autre direction

Le produit scalaire $\mathbf{a} \cdot \mathbf{b}$ pourrait être petit (car $\|\mathbf{b}\|$ est petit), alors que ces deux vecteurs représentent le même concept. La similarité cosinus, elle, donnera 1 dans les deux cas.

### Valeurs

| $\cos\theta$ | Interprétation |
|---|---|
| $1$ | Même direction exactement |
| $0$ | Perpendiculaires (aucune relation) |
| $-1$ | Directions opposées |

### Lien avec l'attention

Dans le mécanisme d'attention, on utilise le **produit scalaire** (pas la similarité cosinus) entre query et key. Pourquoi pas cosinus ? Parce que les normes portent aussi de l'information utile, et parce qu'on va corriger le problème d'échelle autrement (avec le facteur $\sqrt{d_k}$ qu'on verra en Partie 3).

In [ ]:
# Similarité cosinus
a = torch.tensor([3.0, 4.0])
b = torch.tensor([6.0, 8.0])    # même direction que a, mais 2x plus long
c = torch.tensor([-4.0, 3.0])   # perpendiculaire à a

cos_ab = F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0))
cos_ac = F.cosine_similarity(a.unsqueeze(0), c.unsqueeze(0))

print(f"cos(a, b) = {cos_ab.item():.4f}")  # = 1.0 (même direction)
print(f"cos(a, c) = {cos_ac.item():.4f}")  # = 0.0 (perpendiculaires)

# Le produit scalaire, lui, change avec la norme :
print(f"a · b = {torch.dot(a, b).item()}")  # = 50
print(f"a · a = {torch.dot(a, a).item()}")  # = 25 (même direction mais norme différente)

## 0.4 — Matrices

### Définition

Une **matrice** $A \in \mathbb{R}^{m \times n}$ est un tableau rectangulaire de nombres avec $m$ lignes et $n$ colonnes :

$$A = \begin{pmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \end{pmatrix}$$

$a_{ij}$ désigne l'élément à la ligne $i$ et la colonne $j$.

### Multiplication matrice-vecteur

Si $A \in \mathbb{R}^{m \times n}$ et $\mathbf{x} \in \mathbb{R}^n$, alors $\mathbf{y} = A\mathbf{x} \in \mathbb{R}^m$ avec :

$$y_i = \sum_{j=1}^{n} a_{ij} \, x_j$$

Autrement dit, chaque composante de $\mathbf{y}$ est le **produit scalaire** de la $i$-ème ligne de $A$ avec $\mathbf{x}$.

**Exemple** :

$$\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix} \begin{pmatrix} 5 \\ 6 \end{pmatrix} = \begin{pmatrix} 1 \times 5 + 2 \times 6 \\ 3 \times 5 + 4 \times 6 \end{pmatrix} = \begin{pmatrix} 17 \\ 39 \end{pmatrix}$$

### Interprétation comme transformation linéaire

$A\mathbf{x}$ transforme le vecteur $\mathbf{x}$ en un nouveau vecteur $\mathbf{y}$. La matrice $A$ définit la transformation : rotation, étirement, projection, etc. C'est exactement ce que fait une couche linéaire dans un réseau de neurones.

### Multiplication matrice-matrice

Si $A \in \mathbb{R}^{m \times p}$ et $B \in \mathbb{R}^{p \times n}$, alors $C = AB \in \mathbb{R}^{m \times n}$ avec :

$$C_{ij} = \sum_{k=1}^{p} A_{ik} \, B_{kj}$$

**Règle de compatibilité** : le nombre de colonnes de $A$ doit être égal au nombre de lignes de $B$.

**Exemple complet** (2×3 par 3×2) :

$$A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}, \quad B = \begin{pmatrix} 7 & 8 \\ 9 & 10 \\ 11 & 12 \end{pmatrix}$$

$$C_{11} = 1 \times 7 + 2 \times 9 + 3 \times 11 = 7 + 18 + 33 = 58$$
$$C_{12} = 1 \times 8 + 2 \times 10 + 3 \times 12 = 8 + 20 + 36 = 64$$
$$C_{21} = 4 \times 7 + 5 \times 9 + 6 \times 11 = 28 + 45 + 66 = 139$$
$$C_{22} = 4 \times 8 + 5 \times 10 + 6 \times 12 = 32 + 50 + 72 = 154$$

$$C = \begin{pmatrix} 58 & 64 \\ 139 & 154 \end{pmatrix}$$

**Propriétés importantes** :
- **NON commutatif** : $AB \neq BA$ en général (et souvent les dimensions ne sont même pas compatibles)
- **Associatif** : $(AB)C = A(BC)$

### Transposée

La **transposée** $A^T$ s'obtient en échangeant lignes et colonnes : $(A^T)_{ij} = A_{ji}$.

Si $A$ est $(m \times n)$, alors $A^T$ est $(n \times m)$.

Propriété clé : $(AB)^T = B^T A^T$ (l'ordre s'inverse).

### Complexité de la multiplication

Pour calculer $C = AB$ avec $A \in \mathbb{R}^{m \times p}$ et $B \in \mathbb{R}^{p \times n}$ :
- On a $m \times n$ éléments à calculer dans $C$
- Chaque élément $C_{ij}$ nécessite $p$ multiplications et $p-1$ additions
- Total : $m \times n \times p$ multiplications

Donc la complexité est $O(m \cdot n \cdot p)$. Cas particulier : deux matrices carrées $n \times n$ → $O(n^3)$.

In [ ]:
# Multiplication matrice-matrice
A = torch.tensor([[1., 2., 3.],
                   [4., 5., 6.]])
B = torch.tensor([[7., 8.],
                   [9., 10.],
                   [11., 12.]])

C = A @ B
print("A (2x3):")
print(A)
print("\nB (3x2):")
print(B)
print("\nC = A @ B (2x2):")
print(C)  # [[58, 64], [139, 154]]

# Transposée
print("\nA^T:")
print(A.T)

# Non-commutativité : B @ A est (3×3), pas (2×2)
print("\nB @ A (3x3) — résultat complètement différent :")
print(B @ A)

## 0.5 — Fonction exponentielle et softmax

### La fonction exponentielle

La fonction $\exp(x) = e^x$ (où $e \approx 2.718$) a des propriétés qui la rendent centrale en deep learning :

1. $\exp(x) > 0$ pour tout $x \in \mathbb{R}$ → transforme n'importe quel nombre en un nombre **positif**
2. $\exp$ est **strictement croissante** : si $a > b$ alors $\exp(a) > \exp(b)$ → préserve l'ordre
3. $\exp(a + b) = \exp(a) \cdot \exp(b)$ → transforme les sommes en produits
4. $\exp(0) = 1$

Pourquoi $\exp$ et pas une autre fonction positive (comme $x^2$, ou $|x|$) ? Parce que $\exp$ a de bonnes propriétés pour le calcul des gradients : sa dérivée est elle-même ($\frac{d}{dx}\exp(x) = \exp(x)$), ce qui simplifie beaucoup les calculs.

### Le softmax

**Définition.** Soit $\mathbf{z} = (z_1, z_2, \ldots, z_n) \in \mathbb{R}^n$. Le softmax de $\mathbf{z}$ est le vecteur :

$$\text{softmax}(\mathbf{z})_i = \frac{\exp(z_i)}{\displaystyle\sum_{j=1}^{n} \exp(z_j)}$$

### Propriétés (preuves)

**Propriété 1 : La somme vaut 1.**

$$\sum_{i=1}^{n} \text{softmax}(\mathbf{z})_i = \sum_{i=1}^{n} \frac{\exp(z_i)}{\sum_{j} \exp(z_j)} = \frac{\sum_{i} \exp(z_i)}{\sum_{j} \exp(z_j)} = 1 \qquad \blacksquare$$

**Propriété 2 : Chaque valeur est dans $]0, 1[$ (pour $n \geq 2$).** Comme $\exp(z_i) > 0$ et $\sum_j \exp(z_j) > \exp(z_i)$ (car les autres termes sont strictement positifs), on a $0 < \text{softmax}(\mathbf{z})_i < 1$. $\blacksquare$

Donc **le softmax transforme un vecteur de scores quelconques en une distribution de probabilité**.

**Propriété 3 : Monotonie.** Si $z_i > z_k$ alors $\exp(z_i) > \exp(z_k)$ (car $\exp$ est croissante), donc $\text{softmax}(\mathbf{z})_i > \text{softmax}(\mathbf{z})_k$. L'ordre est préservé. $\blacksquare$

**Propriété 4 : Invariance par translation.** Pour tout $c \in \mathbb{R}$ :

$$\text{softmax}(\mathbf{z} + c)_i = \frac{\exp(z_i + c)}{\sum_j \exp(z_j + c)} = \frac{\exp(z_i) \cdot \exp(c)}{\sum_j \exp(z_j) \cdot \exp(c)} = \frac{\exp(z_i)}{\sum_j \exp(z_j)} = \text{softmax}(\mathbf{z})_i$$

On a utilisé $\exp(a + b) = \exp(a)\exp(b)$ et le fait que $\exp(c)$ se simplifie au numérateur et au dénominateur. $\blacksquare$

Cette propriété est utile en pratique : on soustrait $\max(\mathbf{z})$ avant de calculer $\exp$ pour éviter les dépassements numériques (overflow).

### Exemple numérique

$\mathbf{z} = (2.0, \; 1.0, \; 0.1)$ :

| $i$ | $z_i$ | $\exp(z_i)$ | $\text{softmax}(z)_i$ |
|---|---|---|---|
| 1 | 2.0 | 7.389 | 7.389 / 11.212 = 0.659 |
| 2 | 1.0 | 2.718 | 2.718 / 11.212 = 0.242 |
| 3 | 0.1 | 1.105 | 1.105 / 11.212 = 0.099 |

Somme des $\exp$ : $7.389 + 2.718 + 1.105 = 11.212$. Somme des softmax : $0.659 + 0.242 + 0.099 = 1.000$.

### Comportement avec des grandes valeurs (saturation)

Si $\mathbf{z} = (10, 0, 0)$ :
- $\exp(10) \approx 22026$, $\exp(0) = 1$
- $\text{softmax} \approx (22026/22028, \; 1/22028, \; 1/22028) \approx (0.9999, \; 0.00005, \; 0.00005)$

Le softmax **sature** : une seule composante prend presque toute la masse. On verra que c'est exactement ce qui se passe dans l'attention quand $d_k$ est grand et qu'on ne divise pas par $\sqrt{d_k}$.

### Température

On peut contrôler la "dureté" du softmax avec un paramètre de **température** $\tau > 0$ :

$$\text{softmax}(\mathbf{z} / \tau)_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

- $\tau \to 0$ : le softmax devient un **argmax** (one-hot sur le plus grand $z_i$)
- $\tau \to \infty$ : le softmax devient **uniforme** $(1/n, \ldots, 1/n)$
- $\tau = 1$ : softmax standard

On verra que dans l'attention, $\sqrt{d_k}$ joue exactement le rôle d'une température.

In [ ]:
# Softmax à la main vs PyTorch
z = torch.tensor([2.0, 1.0, 0.1])

# À la main
exp_z = torch.exp(z)
softmax_manual = exp_z / exp_z.sum()
print("Softmax (à la main) :", softmax_manual)
print("Somme              :", softmax_manual.sum().item())  # = 1.0

# Avec PyTorch
softmax_torch = F.softmax(z, dim=0)
print("Softmax (torch)    :", softmax_torch)

# Saturation avec grandes valeurs
z_big = torch.tensor([10.0, 0.0, 0.0])
print("\nSoftmax([10, 0, 0]) :", F.softmax(z_big, dim=0))  # ≈ [1, 0, 0]

# Effet de la température
print("\nEffet de la température sur z = [2, 1, 0.1] :")
for tau in [0.1, 0.5, 1.0, 2.0, 10.0]:
    s = F.softmax(z / tau, dim=0)
    print(f"  τ = {tau:4.1f}  →  softmax = [{s[0]:.4f}, {s[1]:.4f}, {s[2]:.4f}]")

## 0.6 — Espérance, variance, indépendance

On a besoin de ces notions pour la preuve du facteur $\sqrt{d_k}$ (Partie 3). Pas besoin de tout le cours de proba, juste les outils essentiels.

### Variable aléatoire

Une **variable aléatoire** $X$ c'est un nombre dont la valeur dépend du hasard. Par exemple : le résultat d'un dé, ou la valeur d'un pixel bruité.

### Espérance

L'**espérance** $\mathbb{E}[X]$ c'est la moyenne de $X$ si on la tirait un grand nombre de fois.

**Cas discret** : si $X$ prend les valeurs $x_1, \ldots, x_k$ avec probabilités $p_1, \ldots, p_k$ :

$$\mathbb{E}[X] = \sum_{i=1}^{k} p_i \, x_i$$

**Exemple** : un dé à 6 faces. $\mathbb{E}[X] = \frac{1}{6}(1 + 2 + 3 + 4 + 5 + 6) = 3.5$.

### Propriété fondamentale : linéarité de l'espérance

Pour toutes variables aléatoires $X, Y$ et constantes $a, b \in \mathbb{R}$ :

$$\mathbb{E}[aX + bY] = a\,\mathbb{E}[X] + b\,\mathbb{E}[Y]$$

**Preuve** (cas discret, l'idée est la même en continu) :

$$\mathbb{E}[aX + bY] = \sum_{i,j} p_{ij} \, (a\,x_i + b\,y_j) = a \sum_{i,j} p_{ij} \, x_i + b \sum_{i,j} p_{ij} \, y_j = a \sum_i p_i \, x_i + b \sum_j p_j \, y_j = a\,\mathbb{E}[X] + b\,\mathbb{E}[Y]$$

où $p_{ij}$ est la probabilité conjointe, et on utilise $\sum_j p_{ij} = p_i$ (probabilité marginale). $\blacksquare$

**Remarque** : la linéarité marche **toujours**, même si $X$ et $Y$ ne sont pas indépendants.

### Variance

La **variance** mesure à quel point $X$ s'éloigne de sa moyenne :

$$\text{Var}(X) = \mathbb{E}\big[(X - \mathbb{E}[X])^2\big]$$

**Formule alternative** (très utile) :

$$\text{Var}(X) = \mathbb{E}[X^2] - \big(\mathbb{E}[X]\big)^2$$

**Preuve** :

$$\text{Var}(X) = \mathbb{E}\big[(X - \mathbb{E}[X])^2\big] = \mathbb{E}\big[X^2 - 2X\,\mathbb{E}[X] + (\mathbb{E}[X])^2\big]$$

Par linéarité de l'espérance :

$$= \mathbb{E}[X^2] - 2\,\mathbb{E}[X]\,\mathbb{E}[X] + (\mathbb{E}[X])^2 = \mathbb{E}[X^2] - (\mathbb{E}[X])^2 \qquad \blacksquare$$

L'**écart-type** est $\sigma = \sqrt{\text{Var}(X)}$. Il a la même unité que $X$.

### Propriété : $\text{Var}(aX) = a^2 \, \text{Var}(X)$

**Preuve** :

$$\text{Var}(aX) = \mathbb{E}[(aX)^2] - (\mathbb{E}[aX])^2 = a^2\,\mathbb{E}[X^2] - a^2\,(\mathbb{E}[X])^2 = a^2\big(\mathbb{E}[X^2] - (\mathbb{E}[X])^2\big) = a^2\,\text{Var}(X) \qquad \blacksquare$$

### Indépendance

Deux variables $X$ et $Y$ sont **indépendantes** si connaître la valeur de l'une ne donne aucune information sur l'autre. Formellement : $P(X = x \text{ et } Y = y) = P(X = x) \cdot P(Y = y)$ pour tous $x, y$.

**Conséquence clé** : si $X$ et $Y$ sont indépendantes, alors :

$$\mathbb{E}[XY] = \mathbb{E}[X] \cdot \mathbb{E}[Y]$$

(On admet cette propriété — la preuve est un calcul direct à partir de la définition de l'espérance conjointe.)

### Propriété : $\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y)$ si $X, Y$ indépendants

**Preuve** :

$$\text{Var}(X + Y) = \mathbb{E}[(X+Y)^2] - (\mathbb{E}[X+Y])^2$$

Développons :

$$\mathbb{E}[(X+Y)^2] = \mathbb{E}[X^2 + 2XY + Y^2] = \mathbb{E}[X^2] + 2\,\mathbb{E}[XY] + \mathbb{E}[Y^2]$$

$$(\mathbb{E}[X+Y])^2 = (\mathbb{E}[X] + \mathbb{E}[Y])^2 = (\mathbb{E}[X])^2 + 2\,\mathbb{E}[X]\,\mathbb{E}[Y] + (\mathbb{E}[Y])^2$$

En soustrayant, et en utilisant $\mathbb{E}[XY] = \mathbb{E}[X]\,\mathbb{E}[Y]$ (indépendance) :

$$\text{Var}(X + Y) = \big(\mathbb{E}[X^2] - (\mathbb{E}[X])^2\big) + \big(\mathbb{E}[Y^2] - (\mathbb{E}[Y])^2\big) = \text{Var}(X) + \text{Var}(Y) \qquad \blacksquare$$

**Par récurrence**, si $X_1, \ldots, X_n$ sont mutuellement indépendants :

$$\text{Var}\left(\sum_{i=1}^{n} X_i\right) = \sum_{i=1}^{n} \text{Var}(X_i)$$

### Distribution normale $\mathcal{N}(\mu, \sigma^2)$

Une variable $X \sim \mathcal{N}(\mu, \sigma^2)$ a :
- Espérance $\mathbb{E}[X] = \mu$
- Variance $\text{Var}(X) = \sigma^2$

Le cas $\mathcal{N}(0, 1)$ s'appelle la **loi normale centrée réduite** : moyenne 0, variance 1.

On va utiliser $\mathcal{N}(0, 1)$ pour modéliser les composantes des vecteurs query et key dans la preuve du $\sqrt{d_k}$.

In [3]:
# Vérification empirique des propriétés de l'espérance et la variance
N = 1_000_000  # un million de tirages

# X ~ N(0, 1), Y ~ N(0, 1) indépendants
X = torch.randn(N)
Y = torch.randn(N)

print("=== Espérance ===")
print(f"E[X]     = {X.mean():.4f}  (théorique : 0)")
print(f"E[3X+2Y] = {(3*X + 2*Y).mean():.4f}  (théorique : 0)")

print("\n=== Variance ===")
print(f"Var(X)   = {X.var():.4f}  (théorique : 1)")
print(f"Var(2X)  = {(2*X).var():.4f}  (théorique : 4 = 2² × 1)")

print("\n=== Indépendance ===")
print(f"E[XY]       = {(X*Y).mean():.4f}  (théorique : E[X]·E[Y] = 0)")
print(f"Var(X + Y)  = {(X+Y).var():.4f}  (théorique : Var(X) + Var(Y) = 2)")

print("\n=== Distribution normale N(0,1) ===")
dans_1sigma = ((X > -1) & (X < 1)).float().mean()
dans_2sigma = ((X > -2) & (X < 2)).float().mean()
print(f"% dans [-1, 1] : {dans_1sigma:.3f}  (théorique : 0.683)")
print(f"% dans [-2, 2] : {dans_2sigma:.3f}  (théorique : 0.954)")

=== Espérance ===
E[X]     = -0.0006  (théorique : 0)
E[3X+2Y] = -0.0023  (théorique : 0)

=== Variance ===
Var(X)   = 1.0000  (théorique : 1)
Var(2X)  = 3.9999  (théorique : 4 = 2² × 1)

=== Indépendance ===
E[XY]       = -0.0022  (théorique : E[X]·E[Y] = 0)
Var(X + Y)  = 1.9957  (théorique : Var(X) + Var(Y) = 2)

=== Distribution normale N(0,1) ===
% dans [-1, 1] : 0.683  (théorique : 0.683)
% dans [-2, 2] : 0.955  (théorique : 0.954)


## 0.7 — Notations Big-O

### Définition

Quand on analyse un algorithme, on veut savoir comment son temps d'exécution **grandit** quand l'entrée grandit. On ignore les constantes multiplicatives.

On dit que $f(n) = O(g(n))$ s'il existe des constantes $C > 0$ et $n_0$ telles que pour tout $n \geq n_0$ :

$$f(n) \leq C \cdot g(n)$$

En gros : $f$ ne grandit pas plus vite que $g$.

### Exemples

| Complexité | Nom | Exemple |
|---|---|---|
| $O(1)$ | Constante | Accéder à un élément d'un tableau |
| $O(n)$ | Linéaire | Parcourir une liste |
| $O(n \log n)$ | Quasi-linéaire | Trier une liste |
| $O(n^2)$ | Quadratique | Comparer tous les paires |
| $O(n^3)$ | Cubique | Multiplication matricielle naïve |

### Application : complexité de la multiplication matricielle

Pour $A \in \mathbb{R}^{m \times p}$ et $B \in \mathbb{R}^{p \times n}$ :
- Résultat : $C \in \mathbb{R}^{m \times n}$ → $m \times n$ éléments à calculer
- Chaque $C_{ij} = \sum_{k=1}^{p} A_{ik} B_{kj}$ → $p$ multiplications

Total : $m \times n \times p$ multiplications → $O(m \cdot n \cdot p)$.

C'est la brique fondamentale : toute l'analyse de complexité du Transformer repose sur le comptage des multiplications matricielles.

---
# Parcours de l'article — section par section

Avant de plonger dans les maths (qu'on a posées en Partie 0), on parcourt le papier pour comprendre **les choix de design** et **le contexte**. Pour chaque section : l'idée centrale et les choix non-évidents.

## §1 — Introduction

**Idée centrale.** Les RNN (LSTM, GRU) dominent la modélisation de séquences, mais leur nature **séquentielle** est une limite fondamentale : on ne peut pas paralléliser le calcul le long de la séquence, et les dépendances longues traversent $O(n)$ étapes. Le papier propose de **se passer complètement de la récurrence** et de ne garder **que l'attention**.

**Choix non-évident.** C'est un pari radical : en 2017, personne n'avait réussi à faire marcher un modèle seq2seq sans aucune récurrence ni convolution. L'attention existait déjà (Bahdanau 2015), mais toujours en complément d'un RNN.

## §2 — Background

**Idée centrale.** ByteNet et ConvS2S utilisent des convolutions pour paralléliser, mais le nombre d'opérations pour relier deux positions distantes **croît avec la distance** (linéairement pour ConvS2S, logarithmiquement pour ByteNet). Le Transformer ramène ce nombre à $O(1)$ — mais au prix d'une résolution effective réduite par le moyennage, que le multi-head vient compenser.

La **self-attention** existait déjà pour des tâches comme la lecture de texte et la classification. Ce qui est nouveau c'est de l'utiliser comme **seul** mécanisme de séquence, sans récurrence du tout.

## §3 — Architecture du modèle

### §3.1 — Encoder / Decoder stacks

- **Encoder** : $N = 6$ couches identiques, chacune = [self-attention multi-têtes] → [FFN], avec résiduelle + LayerNorm autour de chaque sous-couche.
- **Decoder** : $N = 6$ couches, chacune = [self-attention **masquée**] → [cross-attention encoder] → [FFN], même structure résiduelle + LN.
- Toutes les sous-couches produisent des sorties de dimension $d_{\text{model}} = 512$.

**Choix non-évident.** $d_{\text{model}}$ est constant partout (pas de pyramide comme en vision). Ça simplifie les connexions résiduelles : $x + f(x)$ n'est possible que si $x$ et $f(x)$ ont la même dimension.

### §3.2 — Attention

Deux variantes d'attention existaient :
- **Additive** (Bahdanau) : $\text{score}(q, k) = v^T \tanh(W_q q + W_k k)$ — plus expressive mais plus lente
- **Dot-product** : $\text{score}(q, k) = q \cdot k$ — rapide (pur matmul) mais diverge pour grand $d_k$

Le papier choisit le dot-product avec le scaling $1/\sqrt{d_k}$ pour corriger le problème de variance (qu'on a prouvé en Partie 0.6 / Partie 3).

### §3.3 — Position-wise FFN

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

$d_{ff} = 2048 = 4 \times d_{\text{model}}$. C'est là qu'une grosse part des paramètres se trouve (2 × 512 × 2048 ≈ 2M par couche). Appliqué **indépendamment à chaque position**.

### §3.4 — Embeddings and Softmax

**Weight tying** : les trois matrices suivantes **partagent les mêmes poids** :
1. L'embedding source ($\mathbb{R}^{V \times d}$)
2. L'embedding cible ($\mathbb{R}^{V \times d}$)
3. La projection de sortie avant softmax ($\mathbb{R}^{d \times V}$, qui est la transposée de l'embedding cible)

**Pourquoi ça marche** : le logit du token $i$ est $E_i \cdot h$ (produit scalaire entre le vecteur d'embedding de $i$ et l'état caché $h$). C'est une similarité cosinus × normes : le modèle apprend naturellement à produire des états cachés proches de l'embedding du bon token.

Les embeddings sont multipliés par $\sqrt{d_{\text{model}}}$ avant d'ajouter le PE (on a expliqué pourquoi dans l'actuelle section 5.5).

### §3.5 — Positional Encoding

Sinusoïdes fixes vs apprises : les auteurs testent les deux (Table 3, row E). Résultats quasi identiques. Le choix des sinusoïdes est motivé par l'**extrapolation** : le modèle peut traiter des séquences plus longues que celles vues à l'entraînement, car les sinusoïdes sont définies pour tout entier.

## §4 — Why Self-Attention

Trois critères de comparaison (Table 1 du papier) — on les a dérivés en détail dans l'actuelle Partie 6. Le message clé du papier :

> La self-attention est le seul mécanisme qui a **simultanément** $O(1)$ opérations séquentielles et $O(1)$ chemin de dépendance max. Le prix : une complexité $O(n^2 d)$ quadratique en la longueur.

Pour $n < d$ (cas typique en traduction), c'est moins cher que le RNN.

## §5 — Training

### Optimizer

Adam avec $\beta_1 = 0.9$, $\beta_2 = 0.98$ (plus bas que le défaut de 0.999), $\epsilon = 10^{-9}$.

### Learning rate schedule

$$\text{lrate}(\text{step}) = d_{\text{model}}^{-0.5} \cdot \min\!\left(\text{step}^{-0.5}, \; \text{step} \cdot \text{warmup\_steps}^{-1.5}\right)$$

avec $\text{warmup\_steps} = 4000$.

**Analyse** :
- **Phase 1** ($\text{step} < \text{warmup}$) : le $\min$ est atteint par le deuxième terme → $\text{lrate} \propto \text{step}$. **Croissance linéaire** du learning rate.
- **Phase 2** ($\text{step} > \text{warmup}$) : le $\min$ est atteint par le premier terme → $\text{lrate} \propto \text{step}^{-0.5}$. **Décroissance** proportionnelle à l'inverse de la racine carrée.

Le warmup est important : sans lui, les premiers updates (quand les poids sont aléatoires) ont des gradients énormes et l'entraînement diverge.

### Régularisation

- **Residual Dropout** : $P_{\text{drop}} = 0.1$, appliqué sur la sortie de chaque sous-couche avant la résiduelle, et sur les embeddings + PE.
- **Label Smoothing** : $\epsilon_{ls} = 0.1$. Au lieu de cibler une distribution one-hot pour le token correct, on cible $1 - \epsilon$ pour le bon token et $\epsilon / (V-1)$ pour les autres. Ça dégrade la perplexité mais améliore l'accuracy et le BLEU.

In [ ]:
# Learning rate schedule du papier
def lr_schedule(step, d_model=512, warmup=4000):
    if step == 0:
        step = 1
    return d_model ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5))

# Visualisation textuelle
print("Learning rate schedule (d_model=512, warmup=4000) :")
print(f"{'step':>8}  {'lr':>12}")
for step in [1, 100, 500, 1000, 2000, 4000, 8000, 20000, 50000, 100000]:
    lr = lr_schedule(step)
    bar = "█" * int(lr * 30000)
    print(f"{step:8d}  {lr:12.6f}  {bar}")
print(f"\nPic au step {4000} (= warmup)")

## §6 — Results

### Traduction

| Modèle | BLEU EN→DE | BLEU EN→FR | Coût (FLOPs) |
|---|---|---|---|
| GNMT + RL Ensemble | 26.30 | 41.16 | $1.1 \times 10^{21}$ |
| ConvS2S Ensemble | 26.36 | **41.29** | $1.2 \times 10^{21}$ |
| **Transformer (base)** | 27.3 | 38.1 | $\mathbf{3.3 \times 10^{18}}$ |
| **Transformer (big)** | **28.4** | **41.8** | $2.3 \times 10^{19}$ |

Le Transformer (base) atteint des résultats compétitifs en utilisant **~300× moins de compute** que les ensembles. Le big model bat tout.

### Ablation (Table 3)

Points clés :
- **(A)** Single-head = −0.9 BLEU. Trop de têtes (32) dégrade aussi légèrement.
- **(B)** Réduire $d_k$ dégrade → le dot-product n'est pas forcément la meilleure compatibility function.
- **(C)** Plus c'est gros, mieux c'est (scaling law avant l'heure).
- **(D)** Le dropout est crucial, surtout pour les petits modèles.
- **(E)** PE appris ≈ PE sinusoïdal → les deux marchent aussi bien.

### Parsing

Le Transformer se transfère bien au parsing constituant (WSJ), surpassant tous les modèles sauf le RNNG génératif — sans aucun tuning spécifique à la tâche.

---
# Partie 1 — Réseau de neurones : le minimum vital

On a juste besoin de 3 concepts pour comprendre le Transformer : couche linéaire, activation, et paramètres apprenables.

## 1.1 — La couche linéaire

### Définition

Une **couche linéaire** calcule :

$$\mathbf{y} = \mathbf{x} W + \mathbf{b}$$

où :
- $\mathbf{x} \in \mathbb{R}^{d_{\text{in}}}$ est l'entrée (un vecteur ligne)
- $W \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$ est la **matrice de poids**
- $\mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$ est le **biais**
- $\mathbf{y} \in \mathbb{R}^{d_{\text{out}}}$ est la sortie

C'est juste une multiplication matricielle (qu'on a vue en 0.4) plus un vecteur constant.

### Paramètres apprenables

$W$ et $\mathbf{b}$ sont des nombres que le réseau **ajuste pendant l'entraînement**. Au départ ils sont aléatoires, et un algorithme (descente de gradient) les modifie progressivement pour que le réseau fasse ce qu'on veut.

### Exemple

$\mathbf{x} = (1, 2)$, $W = \begin{pmatrix} 0.5 & -0.3 & 0.1 \\ 0.2 & 0.4 & -0.5 \end{pmatrix}$, $\mathbf{b} = (0.1, 0, -0.2)$.

$$y_1 = 1 \times 0.5 + 2 \times 0.2 + 0.1 = 0.5 + 0.4 + 0.1 = 1.0$$
$$y_2 = 1 \times (-0.3) + 2 \times 0.4 + 0 = -0.3 + 0.8 = 0.5$$
$$y_3 = 1 \times 0.1 + 2 \times (-0.5) + (-0.2) = 0.1 - 1.0 - 0.2 = -1.1$$

In [ ]:
# Couche linéaire : à la main vs nn.Linear
x = torch.tensor([[1.0, 2.0]])
W = torch.tensor([[0.5, -0.3, 0.1],
                   [0.2,  0.4, -0.5]])
b = torch.tensor([0.1, 0.0, -0.2])

# À la main
y_manual = x @ W + b
print("y (à la main) :", y_manual)  # [1.0, 0.5, -1.1]

# Avec nn.Linear (attention : PyTorch stocke W transposé)
linear = nn.Linear(2, 3, bias=True)
with torch.no_grad():
    linear.weight.copy_(W.T)  # nn.Linear stocke W^T
    linear.bias.copy_(b)
y_torch = linear(x)
print("y (nn.Linear) :", y_torch)

# Vérification
print("Identiques :", torch.allclose(y_manual, y_torch))

## 1.2 — Fonctions d'activation

### Pourquoi on en a besoin

Sans fonction d'activation, empiler des couches linéaires ne sert à rien.

**Preuve** : si on a deux couches $\mathbf{y}_1 = \mathbf{x} W_1 + \mathbf{b}_1$ et $\mathbf{y}_2 = \mathbf{y}_1 W_2 + \mathbf{b}_2$, alors :

$$\mathbf{y}_2 = (\mathbf{x} W_1 + \mathbf{b}_1) W_2 + \mathbf{b}_2 = \mathbf{x} \underbrace{(W_1 W_2)}_{W_3} + \underbrace{(\mathbf{b}_1 W_2 + \mathbf{b}_2)}_{\mathbf{b}_3}$$

C'est encore une transformation linéaire $\mathbf{y}_2 = \mathbf{x} W_3 + \mathbf{b}_3$. Empiler $n$ couches linéaires = une seule couche linéaire. On ne gagne rien. $\blacksquare$

Pour casser cette linéarité, on applique une fonction non-linéaire entre les couches.

### ReLU

$$\text{ReLU}(x) = \max(0, x) = \begin{cases} x & \text{si } x > 0 \\ 0 & \text{si } x \leq 0 \end{cases}$$

C'est la plus simple des activations, et c'est celle utilisée dans le Transformer (dans le FFN, qu'on verra en Partie 5).

In [ ]:
# Preuve que 2 couches linéaires = 1 couche linéaire
torch.manual_seed(42)
W1 = torch.randn(4, 8)
b1 = torch.randn(8)
W2 = torch.randn(8, 3)
b2 = torch.randn(3)

x = torch.randn(1, 4)

# 2 couches
y1 = x @ W1 + b1
y2 = y1 @ W2 + b2

# 1 couche équivalente
W3 = W1 @ W2
b3 = b1 @ W2 + b2
y_equiv = x @ W3 + b3

print("2 couches linéaires :", y2)
print("1 couche équivalente:", y_equiv)
print("Identiques :", torch.allclose(y2, y_equiv, atol=1e-5))

# ReLU
vals = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("\nReLU de", vals.tolist(), "=", F.relu(vals).tolist())

## 1.3 — Réseau feed-forward

En combinant deux couches linéaires avec un ReLU entre les deux, on obtient un **réseau feed-forward** :

$$\text{FFN}(\mathbf{x}) = \text{ReLU}(\mathbf{x} W_1 + \mathbf{b}_1) \, W_2 + \mathbf{b}_2$$

C'est **exactement** ce que le Transformer utilise dans chaque couche (on le reverra en Partie 5 — c'est l'équation 2 du papier, section 3.3).

Le rôle de ce réseau : transformer chaque vecteur de token de manière non-linéaire, indépendamment des autres tokens.

In [ ]:
# Réseau feed-forward minimal
d_model, d_ff = 4, 8

ffn = nn.Sequential(
    nn.Linear(d_model, d_ff),
    nn.ReLU(),
    nn.Linear(d_ff, d_model)
)

x = torch.randn(1, 3, d_model)  # batch=1, 3 tokens, dim=4
y = ffn(x)
print(f"Entrée  : {x.shape}")  # (1, 3, 4)
print(f"Sortie  : {y.shape}")  # (1, 3, 4) — même shape

---
# Partie 2 — Modéliser des séquences

Maintenant qu'on a les outils mathématiques et les bases des réseaux de neurones, le problème concret : comment traiter des séquences de mots ?

## 2.1 — Séquences et tokens

Un **token**, c'est l'unité de base qu'on traite : un mot, un sous-mot, ou un caractère. Par exemple :

- "Le chat dort" → 3 tokens : ["Le", "chat", "dort"]

Chaque token est converti en un **vecteur** de dimension $d_{\text{model}}$ via un *embedding* (une table de correspondance apprise). Par exemple avec $d_{\text{model}} = 4$ :

$$\text{"Le"} \to (0.2, -0.1, 0.5, 0.3), \quad \text{"chat"} \to (0.8, 0.1, -0.3, 0.6), \quad \text{"dort"} \to (-0.1, 0.4, 0.2, -0.5)$$

Une séquence de $n$ tokens devient une **matrice** $X \in \mathbb{R}^{n \times d_{\text{model}}}$ : chaque ligne est le vecteur d'un token.

## 2.2 — Le problème

On veut que chaque token "sache" ce qui se passe autour de lui. Si je traite "Le", "chat", "dort" indépendamment, je perds toute la structure de la phrase. Il faut un mécanisme pour que l'information circule entre les positions.

Trois approches historiques :
1. **RNN** — traitement séquentiel
2. **Convolutions 1D** — traitement local parallèle
3. **Self-attention** — traitement global parallèle (le Transformer)

## 2.3 — RNN (bref)

Un **Recurrent Neural Network** traite la séquence un token à la fois :

$$\mathbf{h}_t = f(W_h \, \mathbf{h}_{t-1} + W_x \, \mathbf{x}_t + \mathbf{b})$$

- $\mathbf{h}_t$ : état caché au temps $t$ (le "résumé" de tout ce qu'on a vu jusqu'ici)
- $W_h \in \mathbb{R}^{d \times d}$ et $W_x \in \mathbb{R}^{d \times d}$ : matrices de poids
- $f$ : une non-linéarité (tanh, ReLU)

### Problèmes

1. **Séquentiel** : $\mathbf{h}_t$ dépend de $\mathbf{h}_{t-1}$, qui dépend de $\mathbf{h}_{t-2}$, etc. → on ne peut pas paralléliser. Les GPUs modernes sont faits pour le calcul parallèle, donc c'est un énorme gaspillage.

2. **Complexité** : à chaque pas, on fait $W_h \mathbf{h}$ qui coûte $O(d^2)$. Sur $n$ pas → $O(n \cdot d^2)$.

3. **Chemin de dépendance long** : pour que l'info du token 1 atteigne le token $n$, elle doit traverser $n-1$ états → chemin de longueur $O(n)$. En pratique, l'info se dégrade sur de longues distances (vanishing gradient).

In [ ]:
# RNN : le problème du traitement séquentiel
import time

d = 256
n = 100

# Simuler un RNN : boucle séquentielle
h = torch.zeros(d)
W_h = torch.randn(d, d) * 0.01
x_seq = torch.randn(n, d)

start = time.time()
for t in range(n):
    h = torch.tanh(W_h @ h + x_seq[t])
rnn_time = time.time() - start

# Comparaison : une seule multiplication matricielle (ce que fait l'attention)
start = time.time()
_ = x_seq @ x_seq.T  # O(n²d) mais en un seul appel parallèle
attn_time = time.time() - start

print(f"RNN ({n} pas séquentiels)     : {rnn_time*1000:.2f} ms")
print(f"Matmul (1 appel parallèle)  : {attn_time*1000:.2f} ms")
print(f"→ Le RNN est ~{rnn_time/max(attn_time, 1e-9):.0f}× plus lent")

## 2.4 — Convolutions 1D

### Définition

Une **convolution 1D** avec un noyau de taille $k$ calcule, à chaque position $t$ :

$$y_t = \sum_{j=0}^{k-1} W_j \, x_{t+j}$$

où $W_0, \ldots, W_{k-1}$ sont les poids du filtre (des matrices $d \times d$ dans le cas général).

### Avantages

- **Parallélisable** : chaque position est calculée indépendamment → $O(1)$ opérations séquentielles
- **Complexité** : $O(k \cdot n \cdot d^2)$ par couche

### Problème : champ réceptif limité

Un noyau de taille $k$ ne "voit" que $k$ positions voisines. Pour que le token 1 puisse influencer le token $n$, il faut empiler $O(n/k)$ couches (ou $O(\log_k n)$ avec des convolutions dilatées).

C'est mieux qu'un RNN, mais il faut quand même beaucoup de couches pour les longues dépendances.

In [ ]:
# Convolution 1D : champ réceptif limité
# Noyau de taille k=3 : chaque position ne voit que 3 voisins

n_tokens, d = 10, 4
x = torch.randn(1, d, n_tokens)  # Conv1d attend (batch, channels, length)

conv = nn.Conv1d(in_channels=d, out_channels=d, kernel_size=3, padding=1)
y = conv(x)
print(f"Entrée  : {x.shape}  (batch, d, n)")
print(f"Sortie  : {y.shape}  (même shape grâce au padding)")
print(f"Champ réceptif d'UNE couche : 3 tokens")
print(f"Pour couvrir {n_tokens} tokens : il faut ~{n_tokens // 3 + 1} couches empilées")

## 2.5 — Table de complexité comparative

Voici le tableau central de la Section 4 du papier. On le dérive proprement dans la Partie 6, mais voici déjà le résumé :

| Type de couche | Complexité par couche | Opérations séquentielles | Chemin max |
|---|---|---|---|
| Self-Attention | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |
| Récurrent (RNN) | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |
| Convolution | $O(k \cdot n \cdot d^2)$ | $O(1)$ | $O(\log_k n)$ |

- $n$ = longueur de la séquence, $d$ = dimension des représentations, $k$ = taille du noyau convolutif

**Le point clé** : la self-attention a un chemin de longueur $O(1)$ (chaque token voit directement tous les autres) et est parallélisable ($O(1)$ opérations séquentielles). Le prix à payer : une complexité $O(n^2 \cdot d)$ qui est quadratique en la longueur de la séquence.

Pour la traduction typique ($n \approx 30\text{-}100$, $d = 512$), on a $n \ll d$ donc $n^2 d \ll n d^2$ : la self-attention est moins coûteuse que le RNN.

---
# Partie 3 — L'attention : traitement mathématique complet

C'est le cœur du papier et le cœur de ce notebook.

## 3.1 — L'intuition Query-Key-Value

### L'analogie

Imaginons une base de données. Chaque entrée a :
- une **clé** (key) : ce qui décrit l'entrée
- une **valeur** (value) : l'information que l'entrée contient

On arrive avec une **requête** (query) et on cherche les entrées dont la clé correspond le mieux. En recherche classique, on prend le meilleur match. En attention, on prend une **moyenne pondérée** de toutes les valeurs, pondérée par la compatibilité entre la query et chaque key.

### Exemple concret

Prenons la phrase : **"Le chat dort sur le tapis"**.

Le mot **"dort"** a besoin de savoir *qui* dort. Sa **query** encode quelque chose comme "je cherche un sujet, un nom". Le mot **"chat"** a une **key** qui encode "je suis un nom, un sujet potentiel". Le produit scalaire entre query("dort") et key("chat") est élevé → le mot "dort" accorde une forte attention à "chat".

En parallèle, le mot **"sur"** cherche un lieu (query), et **"tapis"** a une key qui dit "je suis un lieu" → forte attention de "sur" vers "tapis".

Chaque mot porte aussi une **value** : l'information sémantique qu'il contient. La sortie de l'attention pour "dort" sera une moyenne pondérée des values de tous les mots, avec un gros poids sur "chat".

### D'où viennent Q, K, V ?

L'entrée du mécanisme d'attention, c'est $X \in \mathbb{R}^{n \times d_{\text{model}}}$ : la matrice des embeddings de la séquence (un vecteur par token, cf. section 2.1).

Si on utilisait $Q = K = V = X$ directement, tous les tokens joueraient le même rôle pour les trois fonctions (chercher, proposer, informer). Le modèle n'aurait **aucun paramètre à apprendre** dans l'attention.

Donc on **projette** $X$ à travers des matrices apprises :

$$Q = X \, W^Q, \quad K = X \, W^K, \quad V = X \, W^V$$

où $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$ sont des **couches linéaires** (section 1.1) — des matrices de poids que le réseau ajuste pendant l'entraînement.

Le même token "chat" donne un vecteur query $\mathbf{q}$ différent de son vecteur key $\mathbf{k}$, parce que $W^Q \neq W^K$. Le réseau apprend à encoder dans Q "les features utiles pour chercher" et dans K "les features utiles pour être trouvé".

### Résumé du flux

$$X \xrightarrow{W^Q} Q, \quad X \xrightarrow{W^K} K, \quad X \xrightarrow{W^V} V \quad \longrightarrow \quad \text{Attention}(Q, K, V)$$

## 3.2 — Dot-product attention pas à pas

### Pourquoi le produit scalaire ?

On a besoin d'une fonction qui mesure "à quel point la key $j$ est pertinente pour la query $i$". On a prouvé en section 0.2 que le produit scalaire $\mathbf{q} \cdot \mathbf{k} = \|\mathbf{q}\|\,\|\mathbf{k}\|\cos\theta$ mesure la similarité directionnelle : si $\mathbf{q}$ et $\mathbf{k}$ pointent dans la même direction, le score est grand ; s'ils sont orthogonaux, le score est nul.

C'est exactement ce qu'on veut. Et en plus, le produit scalaire se calcule comme une multiplication matricielle ($QK^T$), ce qui est ultra-rapide sur GPU.

Il existe d'autres fonctions de compatibilité (attention additive de Bahdanau : $\text{score} = v^T \tanh(W_q q + W_k k)$), mais le dot-product est plus simple et plus rapide à même performance.

### Étape 1 : calculer les scores

On a $n$ tokens. Après projection, $Q \in \mathbb{R}^{n \times d_k}$ et $K \in \mathbb{R}^{n \times d_k}$.

La matrice de scores est :

$$S = Q K^T \in \mathbb{R}^{n \times n}$$

Chaque élément $S_{ij} = \mathbf{q}_i \cdot \mathbf{k}_j$ = produit scalaire entre la query du token $i$ et la key du token $j$.

### Exemple numérique (3 tokens, $d_k = 4$)

$$Q = \begin{pmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 1 & 0 & 0 \end{pmatrix}, \quad K = \begin{pmatrix} 1 & 1 & 0 & 0 \\ 0 & 0 & 1 & 1 \\ 1 & 0 & 1 & 0 \end{pmatrix}$$

Calcul de $S = QK^T$ élément par élément :

$$S_{11} = 1 \cdot 1 + 0 \cdot 1 + 1 \cdot 0 + 0 \cdot 0 = 1$$
$$S_{12} = 1 \cdot 0 + 0 \cdot 0 + 1 \cdot 1 + 0 \cdot 1 = 1$$
$$S_{13} = 1 \cdot 1 + 0 \cdot 0 + 1 \cdot 1 + 0 \cdot 0 = 2$$
$$S_{21} = 0 \cdot 1 + 1 \cdot 1 + 0 \cdot 0 + 1 \cdot 0 = 1$$
$$S_{22} = 0 \cdot 0 + 1 \cdot 0 + 0 \cdot 1 + 1 \cdot 1 = 1$$
$$S_{23} = 0 \cdot 1 + 1 \cdot 0 + 0 \cdot 1 + 1 \cdot 0 = 0$$
$$S_{31} = 1 \cdot 1 + 1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 = 2$$
$$S_{32} = 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 1 + 0 \cdot 1 = 0$$
$$S_{33} = 1 \cdot 1 + 1 \cdot 0 + 0 \cdot 1 + 0 \cdot 0 = 1$$

$$S = \begin{pmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \\ 2 & 0 & 1 \end{pmatrix}$$

**Lecture** : le token 1 a un score de 2 avec le token 3 (forte compatibilité) mais seulement 1 avec les tokens 1 et 2. Le token 2 a un score de 0 avec le token 3 (aucune compatibilité).

In [ ]:
# Vérification du calcul des scores
Q = torch.tensor([[1., 0., 1., 0.],
                   [0., 1., 0., 1.],
                   [1., 1., 0., 0.]])

K = torch.tensor([[1., 1., 0., 0.],
                   [0., 0., 1., 1.],
                   [1., 0., 1., 0.]])

S = Q @ K.T
print("S = Q @ K^T :")
print(S)

## 3.3 — Le facteur $\sqrt{d_k}$ : preuve complète

C'est probablement le point le plus important du papier. Le papier dit (Section 3.2.1, note de bas de page 4) :

> *"To illustrate why the dot products get large, assume that the components of $q$ and $k$ are independent random variables with mean 0 and variance 1. Then their dot product $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has mean 0 and variance $d_k$."*

On va prouver ça rigoureusement, en utilisant les outils de la section 0.6.

### Hypothèses

Soit $\mathbf{q} = (q_1, \ldots, q_{d_k})$ et $\mathbf{k} = (k_1, \ldots, k_{d_k})$ deux vecteurs dont les composantes sont des variables aléatoires **indépendantes** avec :

$$\mathbb{E}[q_i] = 0, \quad \text{Var}(q_i) = 1, \quad \mathbb{E}[k_i] = 0, \quad \text{Var}(k_i) = 1$$

Le produit scalaire est $s = \mathbf{q} \cdot \mathbf{k} = \sum_{i=1}^{d_k} q_i k_i$.

### Calcul de $\mathbb{E}[s]$

Par linéarité de l'espérance (section 0.6) :

$$\mathbb{E}[s] = \mathbb{E}\left[\sum_{i=1}^{d_k} q_i k_i\right] = \sum_{i=1}^{d_k} \mathbb{E}[q_i k_i]$$

Comme $q_i$ et $k_i$ sont indépendants :

$$\mathbb{E}[q_i k_i] = \mathbb{E}[q_i] \cdot \mathbb{E}[k_i] = 0 \cdot 0 = 0$$

Donc : $\boxed{\mathbb{E}[s] = 0}$

### Calcul de $\text{Var}(s)$

Les termes $q_i k_i$ sont indépendants entre eux (car les composantes sont indépendantes entre dimensions). Par la propriété de la section 0.6 :

$$\text{Var}(s) = \text{Var}\left(\sum_{i=1}^{d_k} q_i k_i\right) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i)$$

Calculons $\text{Var}(q_i k_i)$ pour un $i$ quelconque :

$$\text{Var}(q_i k_i) = \mathbb{E}[(q_i k_i)^2] - (\mathbb{E}[q_i k_i])^2$$

On vient de montrer que $\mathbb{E}[q_i k_i] = 0$, donc :

$$\text{Var}(q_i k_i) = \mathbb{E}[q_i^2 \, k_i^2]$$

Par indépendance de $q_i$ et $k_i$ :

$$\mathbb{E}[q_i^2 \, k_i^2] = \mathbb{E}[q_i^2] \cdot \mathbb{E}[k_i^2]$$

Or $\mathbb{E}[q_i^2] = \text{Var}(q_i) + (\mathbb{E}[q_i])^2 = 1 + 0 = 1$ (formule de la section 0.6). De même $\mathbb{E}[k_i^2] = 1$.

Donc $\text{Var}(q_i k_i) = 1 \cdot 1 = 1$.

En sommant sur les $d_k$ termes :

$$\boxed{\text{Var}(s) = d_k}$$

L'écart-type est $\sigma_s = \sqrt{d_k}$.

### Le problème

Si $d_k = 64$ (valeur du papier), l'écart-type des scores est $\sqrt{64} = 8$. Les scores typiques sont dans l'intervalle $[-16, +16]$.

Quand on applique le softmax à de tels scores, il **sature** :

In [ ]:
# Démonstration de la saturation du softmax
print("Softmax de scores de magnitude croissante :")
for scale in [1, 4, 8, 16]:
    z = torch.tensor([scale, 0.0, 0.0], dtype=torch.float)
    s = F.softmax(z, dim=0)
    print(f"  softmax([{scale:2d}, 0, 0]) = [{s[0]:.6f}, {s[1]:.6f}, {s[2]:.6f}]")

Quand les scores sont dans $[-16, +16]$, le softmax met presque toute la masse sur un seul token. Le gradient du softmax dans ces zones saturées est quasi nul → **l'entraînement se bloque**.

### La solution : diviser par $\sqrt{d_k}$

Si on pose $s' = s / \sqrt{d_k}$, alors par la propriété $\text{Var}(aX) = a^2 \text{Var}(X)$ (section 0.6) :

$$\text{Var}(s') = \text{Var}\left(\frac{s}{\sqrt{d_k}}\right) = \frac{1}{d_k} \cdot \text{Var}(s) = \frac{d_k}{d_k} = 1$$

Les scores normalisés ont une variance de 1, indépendamment de $d_k$. Le softmax reçoit des valeurs dans une plage raisonnable et produit des gradients utiles.

### Lien avec la température

On a vu en 0.5 que $\text{softmax}(\mathbf{z}/\tau)$ utilise $\tau$ comme température. Ici, $\sqrt{d_k}$ joue exactement le rôle de la température : il contrôle la "dureté" de la distribution d'attention.

### Formule finale (équation 1 du papier)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# Vérification empirique : Var(q · k) = d_k
N = 100_000

print("Vérification empirique de Var(q·k) = d_k :")
print(f"{'d_k':>6}  {'Var (théo)':>12}  {'Var (emp)':>12}  {'Var après /√d_k':>16}")
print("-" * 52)

for d_k in [4, 16, 64, 256, 512]:
    q = torch.randn(N, d_k)
    k = torch.randn(N, d_k)
    dots = (q * k).sum(dim=1)  # produits scalaires
    dots_scaled = dots / math.sqrt(d_k)
    print(f"{d_k:6d}  {d_k:12.1f}  {dots.var().item():12.2f}  {dots_scaled.var().item():16.4f}")

# Montrer l'effet sur softmax
print("\nEffet sur softmax (d_k = 512, 5 keys) :")
d_k = 512
q = torch.randn(1, d_k)
K = torch.randn(5, d_k)
scores = q @ K.T
scores_scaled = scores / math.sqrt(d_k)
print(f"  Sans scaling  : softmax = {F.softmax(scores, dim=-1).data}")
print(f"  Avec /√d_k    : softmax = {F.softmax(scores_scaled, dim=-1).data}")

## 3.4 — Softmax sur les scores : interprétation probabiliste

Après la division par $\sqrt{d_k}$, on applique le softmax **ligne par ligne** :

$$\alpha_{ij} = \text{softmax}\left(\frac{S_{i,:}}{\sqrt{d_k}}\right)_j = \frac{\exp(S_{ij} / \sqrt{d_k})}{\sum_{l=1}^{n} \exp(S_{il} / \sqrt{d_k})}$$

Chaque ligne $\boldsymbol{\alpha}_i = (\alpha_{i1}, \ldots, \alpha_{in})$ est une **distribution de probabilité** sur les $n$ positions (on l'a prouvé en 0.5 : somme = 1, chaque valeur $\in ]0, 1[$).

**Interprétation** : $\alpha_{ij}$ est le "poids d'attention" que le token $i$ accorde au token $j$. Plus c'est grand, plus le token $j$ est jugé pertinent pour le token $i$.

## 3.5 — Somme pondérée des values

La sortie pour le token $i$ est la **moyenne pondérée** des vecteurs value :

$$\text{output}_i = \sum_{j=1}^{n} \alpha_{ij} \, \mathbf{v}_j$$

En forme matricielle :

$$\text{Output} = A \cdot V$$

où $A \in \mathbb{R}^{n \times n}$ est la matrice d'attention (après softmax) et $V \in \mathbb{R}^{n \times d_v}$ est la matrice des values.

**Cas limites** :
- Si $\alpha$ est **one-hot** (un seul 1, le reste 0) : on sélectionne exactement une value. C'est de l'attention "dure".
- Si $\alpha$ est **uniforme** ($1/n$ partout) : on prend la moyenne de toutes les values. Aucun token n'est privilégié.

## 3.6 — Exemple numérique complet

On reprend $Q$ et $K$ de la section 3.2, et on ajoute $V$ :

$$V = \begin{pmatrix} 10 & 0 \\ 0 & 10 \\ 5 & 5 \end{pmatrix}$$

Ici $d_k = 4$, $d_v = 2$, $\sqrt{d_k} = 2$.

### Étape 1 — Scores $S = QK^T$

Déjà calculé en 3.2 :

$$S = \begin{pmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \\ 2 & 0 & 1 \end{pmatrix}$$

### Étape 2 — Scaling $S' = S / \sqrt{d_k} = S / 2$

$$S' = \begin{pmatrix} 0.5 & 0.5 & 1.0 \\ 0.5 & 0.5 & 0.0 \\ 1.0 & 0.0 & 0.5 \end{pmatrix}$$

### Étape 3 — Softmax ligne par ligne

**Ligne 1** de $S'$ : $(0.5, \; 0.5, \; 1.0)$

$$\exp(0.5) = 1.649, \quad \exp(0.5) = 1.649, \quad \exp(1.0) = 2.718$$

Somme : $1.649 + 1.649 + 2.718 = 6.016$

$$\alpha_1 = \left(\frac{1.649}{6.016}, \; \frac{1.649}{6.016}, \; \frac{2.718}{6.016}\right) = (0.274, \; 0.274, \; 0.452)$$

Vérification : $0.274 + 0.274 + 0.452 = 1.000$ ✓

**Ligne 2** de $S'$ : $(0.5, \; 0.5, \; 0.0)$

$$\exp(0.5) = 1.649, \quad \exp(0.5) = 1.649, \quad \exp(0.0) = 1.000$$

Somme : $1.649 + 1.649 + 1.000 = 4.298$

$$\alpha_2 = (0.384, \; 0.384, \; 0.233)$$

**Ligne 3** de $S'$ : $(1.0, \; 0.0, \; 0.5)$

$$\exp(1.0) = 2.718, \quad \exp(0.0) = 1.000, \quad \exp(0.5) = 1.649$$

Somme : $2.718 + 1.000 + 1.649 = 5.367$

$$\alpha_3 = (0.507, \; 0.186, \; 0.307)$$

Matrice d'attention complète :

$$A = \begin{pmatrix} 0.274 & 0.274 & 0.452 \\ 0.384 & 0.384 & 0.233 \\ 0.507 & 0.186 & 0.307 \end{pmatrix}$$

### Étape 4 — Sortie $= A \cdot V$

**Ligne 1** : le token 1 produit sa sortie en faisant la moyenne pondérée des values :

$$\text{output}_1 = 0.274 \times \begin{pmatrix} 10 \\ 0 \end{pmatrix} + 0.274 \times \begin{pmatrix} 0 \\ 10 \end{pmatrix} + 0.452 \times \begin{pmatrix} 5 \\ 5 \end{pmatrix}$$

$$= \begin{pmatrix} 2.74 \\ 0 \end{pmatrix} + \begin{pmatrix} 0 \\ 2.74 \end{pmatrix} + \begin{pmatrix} 2.26 \\ 2.26 \end{pmatrix} = \begin{pmatrix} 5.00 \\ 5.00 \end{pmatrix}$$

**Interprétation** : le token 1 attend surtout au token 3 (poids 0.452). Le token 3 a la value $(5, 5)$, donc la sortie est tirée vers $(5, 5)$. Mais les tokens 1 et 2 contribuent aussi, avec des values $(10, 0)$ et $(0, 10)$ qui s'annulent mutuellement.

On peut vérifier tout ça avec le code :

In [ ]:
# Exemple numérique complet
Q = torch.tensor([[1., 0., 1., 0.],
                   [0., 1., 0., 1.],
                   [1., 1., 0., 0.]])

K = torch.tensor([[1., 1., 0., 0.],
                   [0., 0., 1., 1.],
                   [1., 0., 1., 0.]])

V = torch.tensor([[10., 0.],
                   [0., 10.],
                   [5., 5.]])

d_k = Q.shape[-1]

# Étape 1 : scores
S = Q @ K.T
print("Étape 1 — Scores S = Q K^T :")
print(S)

# Étape 2 : scaling
S_scaled = S / math.sqrt(d_k)
print(f"\nÉtape 2 — Scores scalés S' = S / √{d_k} :")
print(S_scaled)

# Étape 3 : softmax
A = F.softmax(S_scaled, dim=-1)
print("\nÉtape 3 — Attention A = softmax(S') :")
print(A)
print("Somme par ligne (doit être 1) :", A.sum(dim=-1))

# Étape 4 : sortie
output = A @ V
print("\nÉtape 4 — Sortie = A @ V :")
print(output)

# Interprétation
print("\nInterprétation :")
tokens = ["token 1", "token 2", "token 3"]
for i in range(3):
    weights = [f"{A[i,j]:.3f}" for j in range(3)]
    print(f"  {tokens[i]} : attention = [{', '.join(weights)}]")
    max_j = A[i].argmax().item()
    print(f"    → attend surtout au {tokens[max_j]}")

---
# Partie 4 — Multi-Head Attention

## 4.1 — Pourquoi plusieurs têtes ?

Avec une seule tête d'attention, chaque token produit **une seule** distribution d'attention. Mais dans une phrase, les relations sont multiples :

- Relations **syntaxiques** : sujet-verbe, adjectif-nom
- Relations **sémantiques** : coréférences ("il" → "le chat")
- Relations **positionnelles** : mots adjacents

Une seule distribution de probabilité ne peut pas capter tout ça simultanément. Avec $h$ têtes, chacune opère dans un **sous-espace** différent et peut se spécialiser sur un type de relation.

Le papier (Section 3.2.2) le dit :

> *"Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions."*

## 4.2 — Les projections linéaires

Pour chaque tête $i$ ($i = 1, \ldots, h$), on projette les entrées dans un sous-espace de dimension $d_k = d_{\text{model}} / h$ :

$$Q_i = Q \, W_i^Q, \quad K_i = K \, W_i^K, \quad V_i = V \, W_i^V$$

avec $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$ et $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$.

En pratique, on ne fait pas $h$ projections séparées. On fait **une seule** grosse projection puis on reshape :

1. $Q_{\text{all}} = X \, W^Q$ avec $W^Q \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ → shape $(n, d_{\text{model}})$
2. Reshape $(n, d_{\text{model}}) \to (n, h, d_k)$
3. Transposer → $(h, n, d_k)$

Chaque "tranche" le long de la dimension $h$ correspond à une tête.

## 4.3 — Le split en têtes : trace des shapes

Prenons $B = 1$ (taille de batch), $n = 3$ tokens, $d_{\text{model}} = 8$, $h = 2$ têtes, donc $d_k = 8/2 = 4$.

| Étape | Opération | Shape |
|---|---|---|
| Entrée $X$ | | $(1, 3, 8)$ |
| $Q = X W^Q$ | Projection linéaire | $(1, 3, 8)$ |
| `.view(B, n, h, d_k)` | Reshape | $(1, 3, 2, 4)$ |
| `.transpose(1, 2)` | Permuter axes | $(1, 2, 3, 4)$ |

Maintenant l'axe 1 indexe les têtes. On a 2 sous-tenseurs de shape $(1, 3, 4)$, un par tête. L'attention est calculée **indépendamment** dans chaque tête.

| Étape | Shape |
|---|---|
| Scores = $Q_i K_i^T$ | $(1, 2, 3, 3)$ — une matrice $3 \times 3$ par tête |
| Scores $/\sqrt{d_k}$ | $(1, 2, 3, 3)$ |
| Attention = softmax | $(1, 2, 3, 3)$ |
| Head output = Attn $\times V_i$ | $(1, 2, 3, 4)$ — chaque tête produit des vecteurs de dim 4 |

## 4.4 — Concaténation et projection finale

Après l'attention, on a $h$ sorties de dimension $d_v = d_k$ chacune. On les recombine :

| Étape | Opération | Shape |
|---|---|---|
| `.transpose(1, 2)` | Remettre tokens en axe 1 | $(1, 3, 2, 4)$ |
| `.contiguous().view(B, n, d_{\text{model}})` | Concaténer les têtes | $(1, 3, 8)$ |
| $\times W^O$ | Projection finale | $(1, 3, 8)$ |

$W^O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ mélange l'information des différentes têtes.

**Formule du papier** (Section 3.2.2) :

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \, W^O$$
$$\text{où} \quad \text{head}_i = \text{Attention}(Q W_i^Q, \; K W_i^K, \; V W_i^V)$$

### Coût computationnel

Chaque tête fait de l'attention sur des vecteurs de dimension $d_k = d_{\text{model}}/h$ :
- $h$ têtes × $O(n^2 \cdot d_k)$ = $O(n^2 \cdot d_{\text{model}})$ — même coût qu'une seule tête de dimension $d_{\text{model}}$

Les projections ajoutent $O(n \cdot d_{\text{model}}^2)$ chacune (4 projections : $W^Q, W^K, W^V, W^O$).

In [ ]:
# Multi-head attention : trace complète des shapes
B, n, d_model, h = 1, 3, 8, 2
d_k = d_model // h  # = 4

X = torch.randn(B, n, d_model)

# Projections (une seule matrice par projection)
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)
W_O = nn.Linear(d_model, d_model, bias=False)

Q = W_Q(X)  # (1, 3, 8)
K = W_K(X)
V = W_V(X)

print(f"Après projection : Q.shape = {Q.shape}")

# Reshape et transpose pour séparer les têtes
Q = Q.view(B, n, h, d_k).transpose(1, 2)  # (1, 2, 3, 4)
K = K.view(B, n, h, d_k).transpose(1, 2)
V = V.view(B, n, h, d_k).transpose(1, 2)

print(f"Après split : Q.shape = {Q.shape}  (batch, têtes, tokens, d_k)")

# Attention par tête
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
print(f"Scores : {scores.shape}")  # (1, 2, 3, 3)

attn = F.softmax(scores, dim=-1)
print(f"Attention : {attn.shape}")

head_output = attn @ V
print(f"Sortie par tête : {head_output.shape}")  # (1, 2, 3, 4)

# Recombiner les têtes
concat = head_output.transpose(1, 2).contiguous().view(B, n, d_model)
print(f"Après concat : {concat.shape}")  # (1, 3, 8)

output = W_O(concat)
print(f"Sortie finale : {output.shape}")  # (1, 3, 8)

---
# Partie 5 — Architecture complète du Transformer

## 5.1 — Positional Encoding

### Pourquoi c'est nécessaire

L'attention est **invariante par permutation**. Prouvons-le.

**Proposition.** Si on permute les tokens d'une séquence, la sortie de l'attention est permutée de la même façon.

**Preuve.** Soit $P$ une matrice de permutation (une matrice avec exactement un 1 par ligne et par colonne, les autres entrées étant 0). Permuter les tokens revient à remplacer $Q, K, V$ par $PQ, PK, PV$.

Scores permutés : $(PQ)(PK)^T = PQ K^T P^T = P S P^T$

Le softmax est appliqué ligne par ligne, et permuter les lignes/colonnes ne change pas les valeurs de softmax (on permute les entrées et les sorties de la même façon) :

$$\text{softmax}(P S P^T) = P \, \text{softmax}(S) \, P^T$$

Sortie : $P \, \text{softmax}(S) \, P^T \cdot PV = P \, \text{softmax}(S) \, V = P \cdot \text{Output}$

La sortie est simplement permutée. $\blacksquare$

**Conséquence** : sans information de position, le Transformer ne peut pas distinguer "le chat mange la souris" de "la souris mange le chat". Il faut donc **injecter** la position.

### Les formules sinusoïdales

Le papier (Section 3.5) propose d'ajouter aux embeddings un vecteur de position :

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Posons $\omega_i = 1 / 10000^{2i/d_{\text{model}}}$. Chaque paire de dimensions $(2i, 2i+1)$ oscille à la fréquence $\omega_i$.

- Pour $i = 0$ : $\omega_0 = 1$ → oscillation rapide (longueur d'onde $2\pi$)
- Pour $i = d_{\text{model}}/2 - 1$ : $\omega \approx 1/10000$ → oscillation très lente (longueur d'onde $\approx 2\pi \times 10000$)

Chaque position a un "motif" unique, comme un code-barres.

### Preuve : $PE_{pos+k}$ est une transformation linéaire de $PE_{pos}$

C'est la propriété clé qui permet au modèle d'apprendre les positions relatives.

**Proposition.** Pour tout offset $k$ fixé, il existe une matrice $M_k$ (qui ne dépend pas de $pos$) telle que $PE_{pos+k} = M_k \, PE_{pos}$ (composante par composante, par paires de dimensions).

**Preuve.** Fixons une paire de dimensions $(2i, 2i+1)$ et posons $\omega = \omega_i$. On a :

$$PE_{pos} = \begin{pmatrix} \sin(\omega \cdot pos) \\ \cos(\omega \cdot pos) \end{pmatrix}$$

Par les formules d'addition trigonométriques :

$$\sin(\omega(pos + k)) = \sin(\omega \cdot pos)\cos(\omega \cdot k) + \cos(\omega \cdot pos)\sin(\omega \cdot k)$$

$$\cos(\omega(pos + k)) = \cos(\omega \cdot pos)\cos(\omega \cdot k) - \sin(\omega \cdot pos)\sin(\omega \cdot k)$$

En écriture matricielle :

$$\begin{pmatrix} \sin(\omega(pos+k)) \\ \cos(\omega(pos+k)) \end{pmatrix} = \underbrace{\begin{pmatrix} \cos(\omega k) & \sin(\omega k) \\ -\sin(\omega k) & \cos(\omega k) \end{pmatrix}}_{R_k} \begin{pmatrix} \sin(\omega \cdot pos) \\ \cos(\omega \cdot pos) \end{pmatrix}$$

$R_k$ est une **matrice de rotation** d'angle $\omega k$. Elle ne dépend que de $k$ et $\omega$, pas de $pos$.

Le PE complet est la concaténation de $d_{\text{model}}/2$ paires, et chaque paire se transforme indépendamment. Donc la transformation globale est **bloc-diagonale** :

$$M_k = \text{diag}(R_k^{(\omega_0)}, R_k^{(\omega_1)}, \ldots, R_k^{(\omega_{d/2-1})}) \qquad \blacksquare$$

Le modèle peut donc apprendre les positions relatives via une transformation linéaire.

In [ ]:
# Vérification de la propriété de rotation du PE
d_model = 8
pos = 5
k = 3

# Calculer PE(pos) et PE(pos+k)
def compute_pe(position, d):
    pe = torch.zeros(d)
    for i in range(0, d, 2):
        omega = 1.0 / (10000 ** (i / d))
        pe[i] = math.sin(omega * position)
        pe[i+1] = math.cos(omega * position)
    return pe

pe_pos = compute_pe(pos, d_model)
pe_pos_k = compute_pe(pos + k, d_model)

# Construire la matrice de rotation M_k
M_k = torch.zeros(d_model, d_model)
for i in range(0, d_model, 2):
    omega = 1.0 / (10000 ** (i / d_model))
    c = math.cos(omega * k)
    s = math.sin(omega * k)
    M_k[i, i] = c
    M_k[i, i+1] = s
    M_k[i+1, i] = -s
    M_k[i+1, i+1] = c

# Vérifier : M_k @ PE(pos) == PE(pos+k)
pe_transformed = M_k @ pe_pos
print("PE(pos+k) direct  :", pe_pos_k)
print("M_k @ PE(pos)     :", pe_transformed)
print("Identiques :", torch.allclose(pe_pos_k, pe_transformed, atol=1e-6))

# Invariance par permutation de l'attention
print("\n--- Preuve empirique : attention invariante par permutation ---")
torch.manual_seed(42)
X = torch.randn(1, 4, 8)  # 4 tokens, dim 8
Q = K = V = X

scores = Q @ K.transpose(-2, -1) / math.sqrt(8)
attn = F.softmax(scores, dim=-1)
out = attn @ V

# Permuter : échanger tokens 0 et 2
perm = [2, 1, 0, 3]
X_perm = X[:, perm, :]
Q_p = K_p = V_p = X_perm
scores_p = Q_p @ K_p.transpose(-2, -1) / math.sqrt(8)
out_perm = F.softmax(scores_p, dim=-1) @ V_p

print("Sortie originale token 0 :", out[0, 0, :4].data)
print("Sortie permutée token 2  :", out_perm[0, 2, :4].data)
print("Identiques :", torch.allclose(out[0, 0], out_perm[0, 2], atol=1e-6))

## 5.2 — Layer Normalization

### Définition

Pour un vecteur $\mathbf{x} \in \mathbb{R}^d$ :

$$\mu = \frac{1}{d} \sum_{i=1}^{d} x_i \qquad \sigma^2 = \frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2$$

$$\text{LayerNorm}(\mathbf{x})_i = \gamma_i \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta_i$$

- $\gamma, \beta \in \mathbb{R}^d$ : paramètres apprenables (scale et shift)
- $\varepsilon$ : petit nombre pour la stabilité numérique ($\sim 10^{-5}$)

Le résultat a (approximativement, quand $\gamma = 1$ et $\beta = 0$) moyenne 0 et variance 1 **sur la dimension $d$**.

### Pourquoi normaliser

Sans normalisation, les valeurs internes du réseau peuvent grandir ou rétrécir au fil des couches. La normalisation stabilise l'entraînement en gardant les activations dans une plage raisonnable.

### Exemple

$\mathbf{x} = (1, 3, 5, 7)$ :
- $\mu = (1 + 3 + 5 + 7)/4 = 4$
- $\sigma^2 = ((1-4)^2 + (3-4)^2 + (5-4)^2 + (7-4)^2)/4 = (9 + 1 + 1 + 9)/4 = 5$
- $\sigma = \sqrt{5} \approx 2.236$
- Avec $\gamma = 1, \beta = 0$ : $\text{LN}(x_1) = (1 - 4)/\sqrt{5} \approx -1.342$

In [ ]:
# Layer Normalization : à la main vs PyTorch
x = torch.tensor([[1., 3., 5., 7.]])

# À la main
mu = x.mean(dim=-1, keepdim=True)
var = x.var(dim=-1, keepdim=True, unbiased=False)  # variance de population (pas Bessel)
eps = 1e-5
x_norm = (x - mu) / torch.sqrt(var + eps)
print("À la main :")
print(f"  μ = {mu.item():.4f}, σ² = {var.item():.4f}")
print(f"  LN(x) = {x_norm}")

# Avec PyTorch (gamma=1, beta=0 par défaut)
ln = nn.LayerNorm(4, elementwise_affine=False)
print("\nPyTorch :")
print(f"  LN(x) = {ln(x)}")
print(f"  Identiques : {torch.allclose(x_norm, ln(x), atol=1e-5)}")

## 5.3 — Connexions résiduelles

### Formule

Dans le Transformer, chaque sous-couche est enveloppée d'une connexion résiduelle :

$$\text{output} = \text{LayerNorm}(\mathbf{x} + \text{SubLayer}(\mathbf{x}))$$

### Pourquoi ça marche

Le gradient de la sortie par rapport à l'entrée est :

$$\frac{\partial \, \text{output}}{\partial \, \mathbf{x}} = \frac{\partial}{\partial \mathbf{x}} \text{LN}(\mathbf{x} + f(\mathbf{x})) \approx I + \frac{\partial f}{\partial \mathbf{x}}$$

(en simplifiant le LayerNorm pour l'intuition)

Même si $\partial f / \partial \mathbf{x}$ est petit (vanishing gradient), le terme $I$ (matrice identité) garantit que le gradient **passe quand même**. C'est un "raccourci" pour le gradient.

Sans connexion résiduelle, dans un réseau à $N$ couches, le gradient doit traverser $N$ transformations non-linéaires → il peut disparaître exponentiellement. Avec les connexions résiduelles, il y a toujours un chemin direct.

## 5.4 — Feed-Forward Network (Position-wise)

Chaque couche du Transformer contient, en plus de l'attention, un réseau feed-forward (exactement comme en 1.3) :

$$\text{FFN}(\mathbf{x}) = \max(0, \; \mathbf{x} W_1 + \mathbf{b}_1) \, W_2 + \mathbf{b}_2$$

C'est l'**équation 2** du papier.

Dimensions :
- $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{ff}}$ — expansion (512 → 2048 dans le modèle de base)
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{\text{model}}}$ — compression (2048 → 512)

"Position-wise" signifie qu'on applique le même FFN à chaque position **indépendamment**. Les poids sont partagés entre positions, mais pas entre couches.

## 5.5 — Pourquoi multiplier les embeddings par $\sqrt{d_{\text{model}}}$

Le papier (Section 3.4) dit :

> *"In the embedding layers, we multiply those weights by $\sqrt{d_{\text{model}}}$."*

Pourquoi ? C'est une question d'échelle.

L'initialisation par défaut de `nn.Embedding` dans PyTorch est $\mathcal{N}(0, 1)$ : chaque composante a une variance de 1. La norme d'un vecteur d'embedding est alors $\|\mathbf{e}\| \approx \sqrt{d_{\text{model}}}$ (par la loi des grands nombres appliquée à la somme de $d_{\text{model}}$ carrés de $\mathcal{N}(0,1)$).

Le positional encoding a des composantes $\sin(\cdot)$ et $\cos(\cdot)$ de magnitude $\leq 1$ et de variance $\approx 1/2$, donc sa norme est $\approx \sqrt{d_{\text{model}}/2}$.

Avec cette init, les deux sont déjà au même ordre de grandeur ($\sim \sqrt{d_{\text{model}}}$). Mais dans le papier, les auteurs utilisent une initialisation plus petite (type Xavier, variance $\sim 1/d_{\text{model}}$), ce qui donne une norme d'embedding $\approx 1$. Dans ce cas, le PE ($\sim \sqrt{d_{\text{model}}/2} \approx 16$ pour $d=512$) écrase l'embedding. Le scaling par $\sqrt{d_{\text{model}}}$ ramène l'embedding à $\sqrt{d_{\text{model}}} \approx 23$, comparable au PE.

In [ ]:
# Comparaison des normes : embedding vs positional encoding
d_model = 512
vocab_size = 10000

# Embedding avec init type Xavier (variance ~ 1/d_model), comme dans le papier
emb = nn.Embedding(vocab_size, d_model)
nn.init.normal_(emb.weight, mean=0, std=1/d_model**0.5)

# Prendre quelques vecteurs d'embedding
indices = torch.randint(0, vocab_size, (100,))
emb_vectors = emb(indices)

# PE pour les mêmes positions
pe = torch.zeros(100, d_model)
for pos in range(100):
    for i in range(0, d_model, 2):
        omega = 1.0 / (10000 ** (i / d_model))
        pe[pos, i] = math.sin(omega * pos)
        pe[pos, i+1] = math.cos(omega * pos)

print("Normes moyennes :")
print(f"  Embedding (sans scaling)    : {emb_vectors.norm(dim=-1).mean():.2f}")
print(f"  Embedding (× √d_model)     : {(emb_vectors * math.sqrt(d_model)).norm(dim=-1).mean():.2f}")
print(f"  Positional Encoding         : {pe.norm(dim=-1).mean():.2f}")
print(f"\n→ Sans scaling, le PE (~{pe.norm(dim=-1).mean():.0f}) écrase l'embedding (~{emb_vectors.norm(dim=-1).mean():.0f})")
print(f"→ Avec × √{d_model} = × {math.sqrt(d_model):.1f}, les deux sont au même ordre de grandeur")

## 5.6 — Masques

### Masque de padding

Les séquences dans un batch n'ont pas toutes la même longueur. On complète les plus courtes avec un token spécial `<pad>`. Le masque de padding met $-\infty$ aux positions correspondantes dans les scores d'attention, pour que le softmax leur donne un poids de 0.

### Masque causal

Dans le décodeur, le token à la position $t$ ne doit voir que les tokens aux positions $1, \ldots, t$ (pas le futur). On utilise une matrice **triangulaire inférieure** :

$$M_{ij} = \begin{cases} 0 & \text{si } j \leq i \\ -\infty & \text{si } j > i \end{cases}$$

On ajoute ce masque aux scores avant le softmax : $\text{softmax}(S + M)$. Les positions masquées ($-\infty$) deviennent $\exp(-\infty) = 0$ après softmax.

In [ ]:
# Masques
# Masque causal pour une séquence de longueur 5
T = 5
causal_mask = torch.triu(torch.ones(T, T) * float('-inf'), diagonal=1)
print("Masque causal (5×5) :")
print(causal_mask)

# Vérification : softmax avec masque
scores = torch.ones(1, T, T)  # scores uniformes
scores_masked = scores + causal_mask
attn = F.softmax(scores_masked, dim=-1)
print("\nAttention avec masque causal :")
print(attn[0])  # triangulaire inférieure, chaque ligne somme à 1

---
# Partie 6 — Analyse de complexité détaillée

On dérive proprement chaque entrée de la table de la Section 4 du papier.

## 6.1 — Self-attention : $O(n^2 \cdot d)$

On a $Q, K, V \in \mathbb{R}^{n \times d}$ (en notant $d = d_k = d_v$ pour simplifier).

**Étape 1** : $S = Q K^T$.
- $Q$ est $(n \times d)$, $K^T$ est $(d \times n)$
- Par la section 0.4, le coût est $O(n \cdot d \cdot n) = O(n^2 d)$

**Étape 2** : Softmax.
- Matrice $(n \times n)$ → $n$ lignes de $n$ éléments → $O(n^2)$

**Étape 3** : $A \cdot V$.
- $A$ est $(n \times n)$, $V$ est $(n \times d)$
- Coût : $O(n \cdot n \cdot d) = O(n^2 d)$

**Total** : $O(n^2 d) + O(n^2) + O(n^2 d) = O(n^2 d)$

**Opérations séquentielles** : $O(1)$ — tout est des multiplications matricielles, parallélisables.

**Chemin max** : $O(1)$ — chaque token attend directement à tous les autres.

## 6.2 — RNN : $O(n \cdot d^2)$

À chaque pas de temps $t$ :

$$\mathbf{h}_t = f(W_h \, \mathbf{h}_{t-1} + W_x \, \mathbf{x}_t)$$

- $W_h \in \mathbb{R}^{d \times d}$ : la multiplication $W_h \mathbf{h}$ coûte $O(d^2)$
- $W_x \in \mathbb{R}^{d \times d}$ : idem, $O(d^2)$
- Total par pas : $O(d^2)$

Sur $n$ pas **séquentiels** : $O(n \cdot d^2)$.

**Opérations séquentielles** : $O(n)$ — chaque pas dépend du précédent.

**Chemin max** : $O(n)$ — le signal du token 1 doit traverser $n-1$ états pour atteindre le token $n$.

## 6.3 — Convolution 1D : $O(k \cdot n \cdot d^2)$

Noyau de taille $k$. À chaque position $t$ :

$$\mathbf{y}_t = \sum_{j=0}^{k-1} W_j \, \mathbf{x}_{t+j}$$

- Chaque $W_j \mathbf{x}$ coûte $O(d^2)$
- $k$ termes → $O(k \cdot d^2)$ par position
- $n$ positions → $O(k \cdot n \cdot d^2)$

**Opérations séquentielles** : $O(1)$ par couche — toutes les positions en parallèle.

**Chemin max** : une couche a un champ réceptif de $k$. Pour couvrir $n$ positions :
- Convolutions contiguës : $O(n/k)$ couches
- Convolutions dilatées : $O(\log_k n)$ couches

## 6.4 — Quand self-attention gagne vs perd

La self-attention est moins coûteuse que le RNN quand :

$$n^2 \cdot d < n \cdot d^2 \implies n < d$$

Pour le Transformer de base : $d = d_{\text{model}} = 512$. Donc pour $n < 512$ tokens, la self-attention est **moins** coûteuse. En traduction ($n \approx 30$–$100$), c'est largement le cas.

Pour de très longues séquences ($n \gg d$), la self-attention devient **plus** coûteuse que le RNN. C'est pour ça qu'il existe des variantes comme Longformer, Linformer, etc.

### Mémoire

La self-attention doit stocker la matrice d'attention $A \in \mathbb{R}^{n \times n}$ (par tête, par couche) :
- $n = 512$ : $512^2 = 262\text{K}$ entrées — négligeable
- $n = 4096$ : $4096^2 = 16.8\text{M}$ entrées — ça commence à peser
- $n = 32768$ : $32768^2 \approx 10^9$ entrées — problématique

In [ ]:
# Point de croisement self-attention vs RNN
print("Coût (en unités arbitraires) : self-attention = n²d, RNN = nd²")
print(f"{'n':>6}  {'d':>6}  {'Self-Attn (n²d)':>16}  {'RNN (nd²)':>12}  {'Gagnant':>10}")
print("-" * 56)
d = 512
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    sa = n * n * d
    rnn = n * d * d
    winner = "Self-Attn" if sa < rnn else "RNN"
    print(f"{n:6d}  {d:6d}  {sa:16,}  {rnn:12,}  {winner:>10}")

---
# Partie 7 — Exemple complet : trace d'un encoder layer

On trace **chaque** calcul d'une couche d'encoder avec de vrais nombres.

Paramètres : $n = 3$ tokens ("le", "chat", "dort"), $d_{\text{model}} = 4$, $h = 2$ têtes, $d_k = 2$, $d_{ff} = 8$.

## 7.1 — Initialisation

On fixe des valeurs simples pour tous les poids, pour que les calculs soient traçables à la main.

In [ ]:
# === PARAMÈTRES ===
n, d_model, h, d_k, d_ff = 3, 4, 2, 2, 8
torch.manual_seed(0)

# Embeddings (inventés, petits nombres)
E = torch.tensor([[ 0.5,  0.1, -0.3,  0.2],   # "le"
                   [ 0.8, -0.2,  0.4,  0.1],   # "chat"
                   [-0.1,  0.6, -0.1, -0.4]], dtype=torch.float)  # "dort"

# Scaling par sqrt(d_model)
E_scaled = E * math.sqrt(d_model)
print("Embeddings scalés (× √4 = ×2) :")
print(E_scaled)

# Positional encoding (d_model = 4 → 2 paires sin/cos)
PE = torch.zeros(n, d_model)
for pos in range(n):
    for i in range(0, d_model, 2):
        omega = 1.0 / (10000 ** (i / d_model))
        PE[pos, i] = math.sin(omega * pos)
        PE[pos, i+1] = math.cos(omega * pos)

print("\nPositional Encoding :")
print(PE)

# Input = embedding scalé + PE
X = E_scaled + PE
print("\nX = E × √d_model + PE :")
print(X)

## 7.2 — Multi-Head Self-Attention

In [ ]:
# Poids de projection (fixés pour reproductibilité)
torch.manual_seed(1)
W_Q = torch.randn(d_model, d_model) * 0.5
W_K = torch.randn(d_model, d_model) * 0.5
W_V = torch.randn(d_model, d_model) * 0.5
W_O = torch.randn(d_model, d_model) * 0.5

# Projections
Q = X @ W_Q  # (3, 4)
K = X @ W_K
V = X @ W_V
print("Q ="); print(Q)
print("K ="); print(K)
print("V ="); print(V)

# Split en 2 têtes : (3, 4) → (3, 2, 2) → (2, 3, 2)
Q_heads = Q.view(n, h, d_k).transpose(0, 1)  # (2, 3, 2)
K_heads = K.view(n, h, d_k).transpose(0, 1)
V_heads = V.view(n, h, d_k).transpose(0, 1)

print(f"\nAprès split — Q_heads.shape = {Q_heads.shape}  (têtes, tokens, d_k)")

for head_idx in range(h):
    print(f"\n{'='*50}")
    print(f"TÊTE {head_idx + 1}")
    print(f"{'='*50}")

    q_h = Q_heads[head_idx]  # (3, 2)
    k_h = K_heads[head_idx]
    v_h = V_heads[head_idx]

    # Scores
    scores = q_h @ k_h.T  # (3, 3)
    print(f"Scores (q @ k^T) :")
    print(scores)

    # Scaling
    scores_scaled = scores / math.sqrt(d_k)
    print(f"Scores / √{d_k} :")
    print(scores_scaled)

    # Softmax
    attn = F.softmax(scores_scaled, dim=-1)
    print(f"Attention (softmax) :")
    print(attn)

    # Sortie
    head_out = attn @ v_h
    print(f"Sortie tête :")
    print(head_out)

# Recombiner
head_outputs = []
for head_idx in range(h):
    q_h = Q_heads[head_idx]
    k_h = K_heads[head_idx]
    v_h = V_heads[head_idx]
    scores = q_h @ k_h.T / math.sqrt(d_k)
    attn = F.softmax(scores, dim=-1)
    head_outputs.append(attn @ v_h)

# Concat : (3, 2) + (3, 2) → (3, 4)
concat = torch.cat(head_outputs, dim=-1)
print("\nConcat des têtes :", concat.shape)
print(concat)

# Projection finale
attn_output = concat @ W_O
print("\nSortie après W_O :")
print(attn_output)

## 7.3 — Résiduelle + LayerNorm

In [ ]:
# Connexion résiduelle + LayerNorm
eps = 1e-5
residual_1 = X + attn_output
print("X + attn_output (résiduelle) :")
print(residual_1)

# LayerNorm (sans paramètres apprenables pour simplifier)
mu = residual_1.mean(dim=-1, keepdim=True)
var = residual_1.var(dim=-1, keepdim=True, unbiased=False)
eps = 1e-5
X_1 = (residual_1 - mu) / torch.sqrt(var + eps)
print("\nAprès LayerNorm :")
print(X_1)
print("Moyenne par token  :", X_1.mean(dim=-1))  # ≈ 0
print("Variance par token :", X_1.var(dim=-1, unbiased=False))  # ≈ 1

## 7.4 — Feed-Forward Network

In [ ]:
# FFN : deux couches linéaires avec ReLU
torch.manual_seed(2)
W1 = torch.randn(d_model, d_ff) * 0.3
b1 = torch.randn(d_ff) * 0.1
W2 = torch.randn(d_ff, d_model) * 0.3
b2 = torch.randn(d_model) * 0.1

# Calcul
hidden = F.relu(X_1 @ W1 + b1)  # (3, 8)
print(f"Après ReLU (shape {hidden.shape}) :")
print(hidden)
print(f"Neurones à zéro (ReLU) : {(hidden == 0).sum().item()} sur {hidden.numel()}")

ffn_output = hidden @ W2 + b2  # (3, 4)
print(f"\nSortie FFN (shape {ffn_output.shape}) :")
print(ffn_output)

## 7.5 — Deuxième résiduelle + LayerNorm

In [ ]:
# Deuxième connexion résiduelle + LayerNorm
eps = 1e-5
residual_2 = X_1 + ffn_output

mu2 = residual_2.mean(dim=-1, keepdim=True)
var2 = residual_2.var(dim=-1, keepdim=True, unbiased=False)
X_2 = (residual_2 - mu2) / torch.sqrt(var2 + eps)

print("Sortie finale de la couche encoder :")
print(X_2)
print(f"\nShape : {X_2.shape}  (3 tokens × d_model=4)")
print("\nCette matrice est passée à la couche encoder suivante (ou au decoder).")

## 7.6 — Récapitulatif du flux de données

```
Entrée (3 × 4)
    │
    ├── Embedding × √d_model + Positional Encoding
    │
    ▼
┌──────────────────────────┐
│ Multi-Head Self-Attention │  ← Q, K, V projetés, split en 2 têtes,
│                          │    attention calculée, concat, W_O
└──────────┬───────────────┘
           │
    ├── + résiduelle (X + attn_output)
    ├── LayerNorm
    │
    ▼
┌──────────────────────────┐
│ Feed-Forward Network     │  ← Linear(4→8) → ReLU → Linear(8→4)
└──────────┬───────────────┘
           │
    ├── + résiduelle
    ├── LayerNorm
    │
    ▼
Sortie (3 × 4)
```

Chaque étape **préserve la shape** $(n \times d_{\text{model}})$. L'attention mélange l'information **entre** les positions, le FFN transforme **chaque position** indépendamment, et les connexions résiduelles + LayerNorm stabilisent le tout.

---
# Partie 8 — Implémentation from scratch

Maintenant qu'on a compris toutes les maths, on code le Transformer de A à Z en PyTorch. Chaque classe correspond à un concept qu'on a vu.

Conventions :
- Convention de masque : **1 = garder**, **0 = ignorer**
- Post-LN (comme dans le papier) : `LayerNorm(x + SubLayer(x))`
- Dropout aux endroits standard

## 8.1 — `ScaledDotProductAttention`

C'est l'équation 1 du papier qu'on a dérivée en Partie 3 :

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        # Q: (..., T_q, d_k), K: (..., T_k, d_k), V: (..., T_k, d_v)
        d_k = Q.size(-1)

        # Étape 1 : scores = Q @ K^T / √d_k   (cf. section 3.2 et 3.3)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

        # Étape 2 : masque (optionnel)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        # Étape 3 : softmax ligne par ligne (cf. section 3.4)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Étape 4 : somme pondérée des values (cf. section 3.5)
        output = attn_weights @ V

        return output, attn_weights

**Smoke test** — vérifier les shapes :

In [ ]:
B, T_q, T_k, d_k, d_v = 2, 5, 7, 16, 16
sdpa = ScaledDotProductAttention(dropout=0.0)
Q = torch.randn(B, T_q, d_k)
K = torch.randn(B, T_k, d_k)
V = torch.randn(B, T_k, d_v)
out, attn = sdpa(Q, K, V)
assert out.shape == (B, T_q, d_v), f"Shape incorrecte : {out.shape}"
assert attn.shape == (B, T_q, T_k), f"Shape incorrecte : {attn.shape}"
print(f"✓ out: {out.shape}, attn: {attn.shape}")

## 8.2 — `MultiHeadAttention`

On projette, on split en têtes, on fait l'attention par tête, on concat, on projette (cf. Partie 4).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout=dropout)

    def _split_heads(self, x):
        # (B, T, d_model) → (B, num_heads, T, d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x):
        # (B, num_heads, T, d_k) → (B, T, d_model)
        B, _, T, _ = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self._split_heads(self.W_Q(Q))
        K = self._split_heads(self.W_K(K))
        V = self._split_heads(self.W_V(V))

        # mask : (B, 1, 1, T_k) ou (B, 1, T_q, T_k) pour broadcast sur les têtes
        out, attn_weights = self.attention(Q, K, V, mask=mask)

        out = self.W_O(self._merge_heads(out))
        return out, attn_weights

**Smoke test** :

In [ ]:
B, T, d_model, h = 2, 10, 64, 8
mha = MultiHeadAttention(d_model, h)
x = torch.randn(B, T, d_model)
out, attn = mha(x, x, x)  # self-attention
assert out.shape == (B, T, d_model)
assert attn.shape == (B, h, T, T)
print(f"✓ out: {out.shape}, attn: {attn.shape}")

## 8.3 — `PositionwiseFeedForward`

L'équation 2 du papier (cf. section 5.4) :

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [ ]:
ffn = PositionwiseFeedForward(d_model=64, d_ff=256)
x = torch.randn(2, 10, 64)
assert ffn(x).shape == (2, 10, 64)
print(f"✓ FFN : {x.shape} → {ffn(x).shape}")

## 8.4 — `PositionalEncoding`

Les sinusoïdes de la section 5.1 :

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{\text{model}}}), \quad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$$

Astuce d'implémentation : au lieu de calculer $10000^{2i/d}$ directement (numériquement instable pour les grands $d$), on calcule $\exp(2i \cdot (-\ln 10000) / d)$.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * -(math.log(10000.0) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer("pe", pe)

    def forward(self, x):
        # x : (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
pe_layer = PositionalEncoding(d_model=16, max_len=100)
x = torch.zeros(1, 20, 16)
out = pe_layer(x)
assert out.shape == (1, 20, 16)
# Vérifier que les lignes sont bien différentes
assert not torch.allclose(out[0, 0], out[0, 1])
print(f"✓ PE : {out.shape}, lignes distinctes : True")

## 8.5 — `EncoderLayer`

Une couche d'encoder (cf. sections 5.2–5.4) :

```
x → [Multi-Head Self-Attention] → + x → LayerNorm → [FFN] → + → LayerNorm → sortie
```

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Sous-couche 1 : self-attention + résiduelle + LayerNorm (Post-LN)
        attn_out, _ = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout1(attn_out))

        # Sous-couche 2 : FFN + résiduelle + LayerNorm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        return x

In [ ]:
enc = EncoderLayer(d_model=64, num_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
assert enc(x).shape == (2, 10, 64)
print(f"✓ EncoderLayer : {x.shape} → {enc(x).shape}")

## 8.6 — `DecoderLayer`

Le decoder a **3** sous-couches :
1. Masked self-attention (ne voit pas le futur)
2. Cross-attention (Q vient du decoder, K et V viennent de l'encoder)
3. FFN

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        # 1. Masked self-attention
        attn_out, _ = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))

        # 2. Cross-attention : Q = decoder, K = V = encoder (memory)
        cross_out, _ = self.cross_attn(x, memory, memory, mask=src_mask)
        x = self.norm2(x + self.dropout2(cross_out))

        # 3. FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out))

        return x

In [ ]:
dec = DecoderLayer(d_model=64, num_heads=8, d_ff=256)
memory = torch.randn(2, 10, 64)  # sortie de l'encoder
tgt = torch.randn(2, 7, 64)       # séquence cible
assert dec(tgt, memory).shape == (2, 7, 64)
print(f"✓ DecoderLayer : tgt {tgt.shape} + memory {memory.shape} → {dec(tgt, memory).shape}")

## 8.7 — Masques utilitaires

Deux types (cf. section 5.6) :
- **Padding** : shape `(B, 1, 1, T)` — 1 = token valide, 0 = padding
- **Causal** : shape `(1, 1, T, T)` — triangulaire inférieure

In [ ]:
def make_pad_mask(seq, pad_idx=0):
    """(B, T) → (B, 1, 1, T). 1 = garder, 0 = ignorer."""
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

def make_causal_mask(T, device=None):
    """(1, 1, T, T) triangulaire inférieure."""
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))
    return mask.unsqueeze(0).unsqueeze(0)

def make_tgt_mask(tgt, pad_idx=0):
    """Combine padding + causal pour le decoder. (B, 1, T, T)."""
    B, T = tgt.size()
    pad = make_pad_mask(tgt, pad_idx)       # (B, 1, 1, T)
    causal = make_causal_mask(T, tgt.device) # (1, 1, T, T)
    return pad & causal  # broadcast → (B, 1, T, T)

In [ ]:
tgt = torch.tensor([[1, 2, 3, 0, 0],
                    [1, 2, 3, 4, 5]])
print("Pad mask:", make_pad_mask(tgt).shape)
print("Causal mask (T=5):")
print(make_causal_mask(5)[0, 0].int())
print("\nTgt mask combiné (batch 0) :")
print(make_tgt_mask(tgt)[0, 0].int())

## 8.8 — Transformer complet

On assemble tout : Encoder + Decoder + projection de sortie.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 dropout=0.1, max_len=5000, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 dropout=0.1, max_len=5000, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads=8, d_ff=2048, num_layers=6, dropout=0.1,
                 max_len=5000, pad_idx=0, tie_weights=False):
        super().__init__()
        self.pad_idx = pad_idx
        self.encoder = Encoder(src_vocab_size, d_model, num_heads, d_ff,
                               num_layers, dropout, max_len, pad_idx)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads, d_ff,
                               num_layers, dropout, max_len, pad_idx)
        self.generator = nn.Linear(d_model, tgt_vocab_size)

        # Weight tying (§3.4) : partager les poids entre embedding cible et projection sortie
        if tie_weights:
            self.generator.weight = self.decoder.embedding.weight

        # Initialisation Xavier (comme dans le papier)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        return self.encoder(src, src_mask)

    def decode(self, tgt, memory, tgt_mask=None, src_mask=None):
        return self.decoder(tgt, memory, tgt_mask, src_mask)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        memory = self.encode(src, src_mask)
        dec_out = self.decode(tgt, memory, tgt_mask, src_mask)
        return self.generator(dec_out)

**Smoke test d'intégration** : on vérifie que les shapes sont correctes sur un forward complet.

In [ ]:
model = Transformer(
    src_vocab_size=100, tgt_vocab_size=100,
    d_model=64, num_heads=4, d_ff=128,
    num_layers=2, dropout=0.0
)

src = torch.randint(1, 100, (2, 12))  # batch=2, 12 tokens source
tgt = torch.randint(1, 100, (2, 9))   # batch=2, 9 tokens cible
src_mask = make_pad_mask(src)
tgt_mask = make_tgt_mask(tgt)

logits = model(src, tgt, src_mask, tgt_mask)
print(f"src: {src.shape}")
print(f"tgt: {tgt.shape}")
print(f"logits: {logits.shape}  (attendu: (2, 9, 100))")
assert logits.shape == (2, 9, 100)

# Nombre de paramètres
n_params = sum(p.numel() for p in model.parameters())
print(f"\nParamètres : {n_params:,}")

## 8.9 — Sanity check : tâche de copie

Le test ultime : on entraîne le Transformer à **recopier** une séquence. Si le code est correct, le modèle doit atteindre une loss quasi nulle en quelques centaines de steps.

In [ ]:
def make_copy_batch(batch_size=32, seq_len=10, vocab_size=11, device="cpu"):
    src = torch.randint(2, vocab_size, (batch_size, seq_len), device=device)
    bos = torch.full((batch_size, 1), 1, dtype=torch.long, device=device)
    tgt = torch.cat([bos, src], dim=1)  # (B, seq_len+1)
    return src, tgt


def train_copy_task(num_steps=2000, lr=5e-4, seq_len=10, vocab_size=11):
    torch.manual_seed(0)
    device = "cpu"

    model = Transformer(
        src_vocab_size=vocab_size, tgt_vocab_size=vocab_size,
        d_model=64, num_heads=4, d_ff=128, num_layers=2,
        dropout=0.0, pad_idx=0
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    model.train()
    for step in range(1, num_steps + 1):
        src, tgt = make_copy_batch(batch_size=32, seq_len=seq_len,
                                    vocab_size=vocab_size, device=device)
        tgt_input = tgt[:, :-1]   # teacher forcing : tout sauf le dernier
        tgt_output = tgt[:, 1:]   # labels : tout sauf le BOS

        src_mask = make_pad_mask(src)
        tgt_mask = make_tgt_mask(tgt_input)

        logits = model(src, tgt_input, src_mask, tgt_mask)
        loss = criterion(logits.reshape(-1, vocab_size), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 500 == 0 or step == 1:
            print(f"Step {step:4d}  loss = {loss.item():.4f}")

    return model

model = train_copy_task()

**Vérification** : le modèle copie-t-il bien ?

In [ ]:
def greedy_decode(model, src, max_len, bos_idx=1, pad_idx=0):
    model.eval()
    with torch.no_grad():
        src_mask = make_pad_mask(src, pad_idx)
        memory = model.encode(src, src_mask)
        B = src.size(0)
        ys = torch.full((B, 1), bos_idx, dtype=torch.long, device=src.device)
        for _ in range(max_len - 1):
            tgt_mask = make_tgt_mask(ys, pad_idx)
            out = model.decode(ys, memory, tgt_mask, src_mask)
            logits = model.generator(out[:, -1:])
            next_token = logits.argmax(dim=-1)
            ys = torch.cat([ys, next_token], dim=1)
        return ys

# Test sur des séquences aléatoires
torch.manual_seed(99)
test_src = torch.randint(2, 11, (5, 10))
predictions = greedy_decode(model, test_src, max_len=11)

print("Source     → Prédiction")
for i in range(5):
    src_str = test_src[i].tolist()
    pred_str = predictions[i, 1:].tolist()  # sans le BOS
    match = "✓" if src_str == pred_str else "✗"
    print(f"  {src_str} → {pred_str}  {match}")

---
# Partie 9 — Exercices conceptuels

Des questions pour vérifier qu'on a bien compris. Les réponses sont cachées sous les balises `<details>`.

### Exercice 1 — Pourquoi $\sqrt{d_k}$ et pas $d_k$ ou $\sqrt[3]{d_k}$ ?

On a prouvé que $\text{Var}(q \cdot k) = d_k$. Donne un argument quantitatif pour expliquer pourquoi on divise par $\sqrt{d_k}$ et pas par une autre fonction de $d_k$.

<details><summary>▶ Réponse</summary>

On veut que la variance des scores soit 1 (pour que le softmax soit dans un régime ni saturé ni uniforme).

$$\text{Var}\left(\frac{q \cdot k}{c}\right) = \frac{\text{Var}(q \cdot k)}{c^2} = \frac{d_k}{c^2}$$

Pour que ça vaille 1 : $c^2 = d_k$, donc $c = \sqrt{d_k}$.

Si on divisait par $d_k$ : $\text{Var} = d_k / d_k^2 = 1/d_k \to 0$ → distribution quasi-uniforme, pas d'attention.

Si on divisait par $\sqrt[3]{d_k}$ : $\text{Var} = d_k / d_k^{2/3} = d_k^{1/3}$ → grandit encore avec $d_k$.

</details>

### Exercice 2 — Multi-head vs single head

Argumente qu'une seule tête de dimension $d_{\text{model}}$ est strictement moins expressive que $h$ têtes de dimension $d_{\text{model}} / h$.

<details><summary>▶ Réponse</summary>

Avec une seule tête, on calcule UNE distribution d'attention : chaque token a UNE façon de pondérer les autres. Avec $h$ têtes, on a $h$ distributions indépendantes. Par exemple :

- Tête 1 pourrait apprendre à lier sujet et verbe
- Tête 2 pourrait apprendre la proximité positionnelle

Avec une seule tête, le modèle est forcé de faire un compromis : on ne peut pas simultanément mettre toute l'attention sur le verbe ET sur le mot adjacent. Avec $h$ têtes, chacune se spécialise.

Le coût total est le même ($O(n^2 d)$), mais l'expressivité est supérieure car on a $h$ sous-espaces indépendants.

</details>

### Exercice 3 — Shape tracing complet

$B = 2$, $T = 16$, $d_{\text{model}} = 512$, $h = 8$, $d_k = 64$.

Donne la shape de chaque tenseur intermédiaire dans le forward de `MultiHeadAttention`.

<details><summary>▶ Réponse</summary>

| Étape | Shape |
|---|---|
| Entrée $X$ | $(2, 16, 512)$ |
| Après $W^Q(X)$ | $(2, 16, 512)$ |
| Après `view(B, T, h, d_k)` | $(2, 16, 8, 64)$ |
| Après `.transpose(1, 2)` | $(2, 8, 16, 64)$ |
| $QK^T$ | $(2, 8, 16, 16)$ |
| Après softmax | $(2, 8, 16, 16)$ |
| $\times V$ | $(2, 8, 16, 64)$ |
| Après `.transpose(1,2)` | $(2, 16, 8, 64)$ |
| Après `.view(B, T, d_{\text{model}})$ | $(2, 16, 512)$ |
| Après $W^O$ | $(2, 16, 512)$ |

</details>

### Exercice 4 — Sans connexion résiduelle

Que se passe-t-il si on retire la connexion résiduelle autour du FFN ?

<details><summary>▶ Réponse</summary>

Sans résiduelle, le gradient doit traverser le FFN complet à chaque couche. Avec $N$ couches, le gradient passe par $N$ FFN en série. Si les poids du FFN ont des valeurs propres $< 1$, le gradient décroît exponentiellement (vanishing gradient). Si $> 1$, il explose.

La connexion résiduelle $y = x + \text{FFN}(x)$ garantit $\partial y / \partial x = I + \partial\text{FFN}/\partial x$, donc même si le terme FFN est petit, le gradient $I$ passe toujours.

En pratique, sans résiduelle, un Transformer à 6 couches devient quasi impossible à entraîner.

</details>

### Exercice 5 — Attention : permutation invariance

Prouve (sans regarder la section 5.1) que si on permute les tokens d'entrée, la sortie de self-attention est permutée de la même façon.

<details><summary>▶ Réponse</summary>

En self-attention, $Q = K = V = X$. Si on permute par $P$ : $X' = PX$.

$$\text{scores}' = (PX)(PX)^T = PX X^T P^T = P \cdot S \cdot P^T$$

Le softmax est appliqué par ligne. Permuter les lignes et colonnes de $S$ ne change pas les valeurs de softmax (on permute les entrées et les sorties de la même façon) :

$$A' = P \cdot A \cdot P^T$$

Sortie : $A' \cdot V' = P A P^T \cdot PV = P A (P^T P) V = P A V = P \cdot \text{output}$.

On utilise $P^T P = I$ car $P$ est une matrice de permutation (orthogonale). $\blacksquare$

</details>

### Exercice 6 — Pre-LN vs Post-LN

Le papier utilise **Post-LN** : $\text{LN}(x + f(x))$.

Les implémentations modernes (GPT, LLaMA) utilisent **Pre-LN** : $x + f(\text{LN}(x))$.

Écris les deux variantes et explique pourquoi Pre-LN est plus stable à l'entraînement.

<details><summary>▶ Réponse</summary>

**Post-LN** (papier) :
```python
x = LayerNorm(x + self_attn(x))
x = LayerNorm(x + ffn(x))
```

**Pre-LN** :
```python
x = x + self_attn(LayerNorm(x))
x = x + ffn(LayerNorm(x))
```

Avec Pre-LN, l'entrée de chaque sous-couche est toujours normalisée (moyenne 0, variance 1), ce qui stabilise les activations. Le chemin résiduel $x$ n'est jamais normalisé, donc le gradient passe directement sans être "écrasé" par le LayerNorm.

Avec Post-LN, le LayerNorm est appliqué APRÈS l'addition résiduelle. Ça peut poser des problèmes de stabilité au début de l'entraînement (les résidus sont grands, le LayerNorm doit les normaliser).

En pratique, Post-LN nécessite un warmup du learning rate plus long. Pre-LN converge plus facilement.

</details>

### Exercice 7 — Complexité mémoire de l'attention

Pour un Transformer avec $L = 12$ couches, $h = 12$ têtes, $n = 4096$ tokens, combien d'éléments faut-il stocker pour les matrices d'attention ?

<details><summary>▶ Réponse</summary>

Chaque tête, dans chaque couche, stocke une matrice $n \times n$ :

$$L \times h \times n^2 = 12 \times 12 \times 4096^2 = 144 \times 16{,}777{,}216 \approx 2.4 \times 10^9 \text{ éléments}$$

En float32 (4 octets) : $\approx 9.7$ Go rien que pour les matrices d'attention. C'est pour ça que les longs contextes sont coûteux en mémoire.

</details>

### Exercice 8 — Masque causal dans la cross-attention ?

Le masque causal s'applique-t-il dans la cross-attention du decoder ? Justifie.

<details><summary>▶ Réponse</summary>

**Non.** Le masque causal empêche de voir le futur de la séquence **cible**. Dans la cross-attention :
- Q vient du decoder (séquence cible)
- K, V viennent de l'encoder (séquence source)

La séquence source est entièrement connue (elle a déjà été encodée). Chaque position du decoder doit pouvoir voir **toute** la source. Donc pas de masque causal ici.

Le masque causal ne s'applique que dans la **self-attention** du decoder, pour que le token $t$ ne voie que les tokens $1, \ldots, t$ de la cible.

</details>

### Exercice 9 — Calcul du nombre de paramètres

Pour le modèle de base du papier ($d_{\text{model}} = 512$, $d_{ff} = 2048$, $h = 8$, $N = 6$ couches, vocab $= 37000$), estime le nombre total de paramètres.

<details><summary>▶ Réponse</summary>

**Par couche encoder** :
- Multi-head attention : $4 \times d_{\text{model}}^2 = 4 \times 512^2 = 1{,}048{,}576$ (W_Q, W_K, W_V, W_O) + biais
- FFN : $2 \times d_{\text{model}} \times d_{ff} = 2 \times 512 \times 2048 = 2{,}097{,}152$ + biais
- LayerNorm : $2 \times 2 \times d_{\text{model}} = 2{,}048$
- Total par couche : $\approx 3.15$M

**6 couches encoder** : $\approx 18.9$M
**6 couches decoder** : $\approx 4.2$M par couche (3 sous-couches : self-attn + cross-attn + FFN) $\times 6 \approx 25.2$M

**Embeddings** : $37000 \times 512 \approx 18.9$M (partagé source/cible/sortie dans le papier)

**Total** : $\approx 63$M paramètres (le papier dit 65M dans la Table 3 — la différence vient des biais et LayerNorm qu'on a ignorés).

</details>

### Exercice 10 — Weight tying

Justifie que partager les poids entre l'embedding cible et la projection de sortie est raisonnable. Y a-t-il un risque ?

<details><summary>▶ Réponse</summary>

**Justification.** L'embedding cible $E \in \mathbb{R}^{V \times d}$ envoie un token $i$ vers $E_i \in \mathbb{R}^d$. La projection de sortie envoie un état caché $h \in \mathbb{R}^d$ vers des logits $W h \in \mathbb{R}^V$, où $W \in \mathbb{R}^{V \times d}$.

Le logit du token $i$ est $W_i \cdot h$ = produit scalaire entre la ligne $i$ de $W$ et $h$. C'est une mesure de similarité : "à quel point $h$ ressemble au vecteur associé au token $i$".

Si $W = E$ (weight tying), le logit de $i$ est $E_i \cdot h$ : on cherche le token dont l'embedding est le plus proche de l'état caché. C'est exactement le comportement souhaité.

**Avantage** : réduit le nombre de paramètres de $V \times d$ (pour le base model : $37000 \times 512 \approx 19$M paramètres en moins).

**Risque** : l'embedding doit servir à deux tâches (représenter les tokens en entrée ET être la cible en sortie). En pratique, ça marche très bien car les deux rôles sont naturellement alignés.

</details>

### Exercice 11 — Shape tracing d'un forward complet

$B = 4$, $T_{\text{src}} = 12$, $T_{\text{tgt}} = 9$, $V = 100$, $d_{\text{model}} = 64$, $h = 8$, $N = 2$.

Trace les shapes de : `src_mask`, `tgt_mask`, `memory`, `out` (sortie decoder), `logits`.

<details><summary>▶ Réponse</summary>

| Tenseur | Shape | Commentaire |
|---|---|---|
| `src` | $(4, 12)$ | Entiers |
| `tgt` | $(4, 9)$ | Entiers |
| `src_mask` | $(4, 1, 1, 12)$ | Padding mask source |
| `tgt_mask` | $(4, 1, 9, 9)$ | Padding + causal combinés |
| `src_emb` | $(4, 12, 64)$ | Après embedding × √d + PE |
| Encoder layer 1 in/out | $(4, 12, 64)$ | Préservé |
| `memory` | $(4, 12, 64)$ | Sortie encoder |
| `tgt_emb` | $(4, 9, 64)$ | Après embedding × √d + PE |
| Decoder self-attn scores | $(4, 8, 9, 9)$ | Par tête, masque causal |
| Decoder cross-attn scores | $(4, 8, 9, 12)$ | Q=decoder, K=encoder |
| `out` (sortie decoder) | $(4, 9, 64)$ | |
| `logits` | $(4, 9, 100)$ | Projection sur le vocabulaire |

</details>

### Exercice 12 — Masque causal à la main

Écris le code PyTorch pour créer un masque causal booléen $(T \times T)$ **sans** `torch.tril`. Utilise `torch.arange` et le broadcasting.

<details><summary>▶ Réponse</summary>

```python
T = 5
idx = torch.arange(T)
causal = idx.unsqueeze(0) <= idx.unsqueeze(1)  # (1, T) <= (T, 1) → (T, T)
```

Comment ça marche :
- `idx.unsqueeze(1)` → colonne : $\begin{pmatrix} 0 \\ 1 \\ 2 \\ 3 \\ 4 \end{pmatrix}$ (les lignes = "je suis à la position...")
- `idx.unsqueeze(0)` → ligne : $(0, 1, 2, 3, 4)$ (les colonnes = "est-ce que je peux voir la position...")
- La comparaison `<=` donne `True` quand la colonne ≤ la ligne, i.e., "la position $j$ est avant ou égale à la position $i$"

```
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])
```

</details>

---
# Partie 10 — Test final : "les yeux fermés"

Réimplémente tout sans regarder le code de la Partie 8. Les `# TODO` sont à remplir. Les smoke tests à la fin vérifient que tout marche.

In [ ]:
# Imports (déjà faits plus haut, mais pour l'autonomie de cette section)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 10.1 — `MyScaledDotProductAttention`

In [ ]:
class MyScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        # TODO : implémenter l'attention scaled dot-product
        # 1. Calculer les scores : Q @ K^T / √d_k
        # 2. Appliquer le masque (si présent) : masked_fill(mask == 0, -inf)
        # 3. Softmax
        # 4. Dropout
        # 5. Multiplier par V
        # Retourner (output, attention_weights)
        raise NotImplementedError("À toi de jouer")

## 10.2 — `MyMultiHeadAttention`

In [ ]:
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TODO : créer W_Q, W_K, W_V, W_O (nn.Linear)
        # TODO : créer l'attention (MyScaledDotProductAttention)
        raise NotImplementedError("À toi de jouer")

    def _split_heads(self, x):
        # TODO : (B, T, d_model) → (B, num_heads, T, d_k)
        raise NotImplementedError

    def _merge_heads(self, x):
        # TODO : (B, num_heads, T, d_k) → (B, T, d_model)
        raise NotImplementedError

    def forward(self, Q, K, V, mask=None):
        # TODO : projeter, split, attention, merge, projeter
        raise NotImplementedError

## 10.3 — `MyPositionwiseFeedForward`

In [ ]:
class MyPositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        # TODO : 2 couches linéaires + dropout
        raise NotImplementedError

    def forward(self, x):
        # TODO : linear1 → relu → dropout → linear2
        raise NotImplementedError

## 10.4 — `MyPositionalEncoding`

In [ ]:
class MyPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # TODO : construire la matrice PE (max_len, d_model) avec sin/cos
        # et la stocker avec register_buffer
        raise NotImplementedError

    def forward(self, x):
        # TODO : ajouter PE aux embeddings + dropout
        raise NotImplementedError

## 10.5 — `MyEncoderLayer`

In [ ]:
class MyEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO : self_attn, ffn, 2 LayerNorm, 2 Dropout
        raise NotImplementedError

    def forward(self, x, src_mask=None):
        # TODO : self-attn → résiduelle → LN → FFN → résiduelle → LN
        raise NotImplementedError

## 10.6 — `MyDecoderLayer`

In [ ]:
class MyDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # TODO : self_attn, cross_attn, ffn, 3 LayerNorm, 3 Dropout
        raise NotImplementedError

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        # TODO : masked self-attn → cross-attn → FFN (avec résiduelles + LN)
        raise NotImplementedError

## 10.7 — Masques utilitaires

In [ ]:
def my_make_pad_mask(seq, pad_idx=0):
    # TODO : (B, T) → (B, 1, 1, T)
    raise NotImplementedError

def my_make_causal_mask(T, device=None):
    # TODO : (1, 1, T, T) triangulaire inférieure
    raise NotImplementedError

def my_make_tgt_mask(tgt, pad_idx=0):
    # TODO : combiner padding + causal
    raise NotImplementedError

## 10.8 — `MyTransformer` complet

In [ ]:
class MyEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 dropout=0.1, max_len=5000, pad_idx=0):
        super().__init__()
        # TODO : embedding, pos_encoding, layers (ModuleList de MyEncoderLayer)
        # N'oublie pas de stocker d_model et pad_idx
        raise NotImplementedError

    def forward(self, src, src_mask=None):
        # TODO : embedding * sqrt(d_model) + PE, puis passer par chaque layer
        raise NotImplementedError


class MyDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 dropout=0.1, max_len=5000, pad_idx=0):
        super().__init__()
        # TODO : embedding, pos_encoding, layers (ModuleList de MyDecoderLayer)
        raise NotImplementedError

    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        # TODO : embedding * sqrt(d_model) + PE, puis passer par chaque layer
        raise NotImplementedError


class MyTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads=8, d_ff=2048, num_layers=6, dropout=0.1,
                 max_len=5000, pad_idx=0, tie_weights=False):
        super().__init__()
        # TODO : encoder, decoder, generator (nn.Linear)
        # TODO : weight tying si tie_weights
        # TODO : Xavier init (for p in self.parameters(): if p.dim() > 1: xavier_uniform_)
        raise NotImplementedError

    def encode(self, src, src_mask=None):
        raise NotImplementedError

    def decode(self, tgt, memory, tgt_mask=None, src_mask=None):
        raise NotImplementedError

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # TODO : encode → decode → generator
        raise NotImplementedError

## 10.9 — Smoke tests

Si tout passe, c'est bon.

In [ ]:
def run_all_my_tests():
    print("=== Tests ===")
    B, T, d_model, h, d_ff = 2, 10, 64, 8, 256

    # 1. ScaledDotProductAttention
    sdpa = MyScaledDotProductAttention()
    Q = torch.randn(B, T, 16)
    K = torch.randn(B, T, 16)
    V = torch.randn(B, T, 16)
    out, attn = sdpa(Q, K, V)
    assert out.shape == (B, T, 16) and attn.shape == (B, T, T)
    print("✓ ScaledDotProductAttention")

    # 2. MultiHeadAttention
    mha = MyMultiHeadAttention(d_model, h)
    x = torch.randn(B, T, d_model)
    out, attn = mha(x, x, x)
    assert out.shape == (B, T, d_model) and attn.shape == (B, h, T, T)
    print("✓ MultiHeadAttention")

    # 3. FFN
    ffn = MyPositionwiseFeedForward(d_model, d_ff)
    assert ffn(x).shape == (B, T, d_model)
    print("✓ PositionwiseFeedForward")

    # 4. PE
    pe = MyPositionalEncoding(d_model)
    z = torch.zeros(B, T, d_model)
    out = pe(z)
    assert out.shape == (B, T, d_model)
    assert not torch.allclose(out[0, 0], out[0, 1])
    print("✓ PositionalEncoding")

    # 5. EncoderLayer
    enc = MyEncoderLayer(d_model, h, d_ff)
    assert enc(x).shape == (B, T, d_model)
    print("✓ EncoderLayer")

    # 6. DecoderLayer
    dec = MyDecoderLayer(d_model, h, d_ff)
    memory = torch.randn(B, T, d_model)
    tgt = torch.randn(B, 7, d_model)
    assert dec(tgt, memory).shape == (B, 7, d_model)
    print("✓ DecoderLayer")

    # 7. Masques
    seq = torch.tensor([[1, 2, 0], [1, 0, 0]])
    pm = my_make_pad_mask(seq)
    assert pm.shape == (2, 1, 1, 3)
    cm = my_make_causal_mask(5)
    assert cm.shape == (1, 1, 5, 5)
    tm = my_make_tgt_mask(seq)
    assert tm.shape == (2, 1, 3, 3)
    print("✓ Masques")

    # 8. Transformer complet
    model = MyTransformer(
        src_vocab_size=50, tgt_vocab_size=50,
        d_model=d_model, num_heads=h, d_ff=d_ff,
        num_layers=2, dropout=0.0, pad_idx=0
    )
    src = torch.randint(1, 50, (B, 12))
    tgt_in = torch.randint(1, 50, (B, 9))
    src_mask = my_make_pad_mask(src)
    tgt_mask = my_make_tgt_mask(tgt_in)
    logits = model(src, tgt_in, src_mask, tgt_mask)
    assert logits.shape == (B, 9, 50)
    print("✓ Transformer complet")

    print("\n=== Tous les tests passent ===")

# Décommenter quand tout est implémenté :
# run_all_my_tests()

---
# Partie 11 — Pièges courants d'implémentation

Les erreurs classiques quand on code un Transformer. Ça fait gagner des heures de debug.

## Piège 1 — Convention du masque inversée

Deux conventions existent :
- **PyTorch `masked_fill`** : `mask == 0` → mettre `-inf`
- **Certaines implémentations** : `mask == True` → ignorer

Choisir une convention et s'y tenir. Ici : **1 = garder, 0 = ignorer**.

## Piège 2 — Oubli du $\sqrt{d_k}$

Sans la division par $\sqrt{d_k}$, le softmax sature (cf. section 3.3). Les gradients sont quasi nuls et le modèle n'apprend pas. Ça compile, ça tourne, mais la loss ne descend pas.

## Piège 3 — Masque mal broadcasté

Le masque de padding a shape `(B, 1, 1, T)`. Si tu le crées en `(B, T)` sans les `unsqueeze`, il ne se broadcast pas correctement sur les `h` têtes et les `T_q` positions.

## Piège 4 — Confusion self-attention / cross-attention

Dans la cross-attention du decoder :
```python
# CORRECT : Q = decoder, K = V = encoder
cross_attn(x, memory, memory, ...)

# BUG : tout vient du decoder → c'est de la self-attention !
cross_attn(x, x, x, ...)
```

## Piège 5 — Mauvais shift du teacher forcing

La cible en entrée du decoder doit être décalée d'un cran par rapport aux labels :
```python
tgt_input  = tgt[:, :-1]   # [BOS, w1, w2, ..., wn-1]
tgt_labels = tgt[:, 1:]    # [w1,  w2, w3, ..., wn]
```
Si tu donnes la même séquence en entrée et en labels, le modèle "triche" en regardant le token qu'il doit prédire.

## Piège 6 — `.contiguous()` avant `.view()`

Après `.transpose()`, le tenseur n'est plus contigu en mémoire :
```python
x = x.transpose(1, 2)   # ok
x = x.view(B, T, -1)    # RuntimeError !
x = x.contiguous().view(B, T, -1)  # ok
```

## Piège 7 — Dropout mal placé

Le dropout doit être appliqué :
1. Sur les **poids d'attention** (après softmax, avant $\times V$)
2. Sur la sortie de chaque **sous-couche** (avant l'addition résiduelle)
3. Sur la somme **embedding + PE**

Oublier le dropout sur les poids d'attention est le bug le plus courant.

## Piège 8 — Positional encoding mal construit

**Le bug.** Confondre les dimensions paires et impaires, ou avoir un broadcast incorrect entre `position` et `div_term`.

**Symptôme.** Le modèle tourne mais les performances sont anormalement basses. Difficile à détecter car le PE est juste un biais additif — le modèle compense partiellement.

**Bugs fréquents** :
```python
# BUG 1 : sin sur les impaires, cos sur les paires (inversé)
pe[:, 1::2] = torch.sin(position * div_term)  # devrait être 0::2
pe[:, 0::2] = torch.cos(position * div_term)  # devrait être 1::2

# BUG 2 : oublier unsqueeze → broadcast erroné
position = torch.arange(max_len)  # (max_len,) au lieu de (max_len, 1)
pe[:, 0::2] = torch.sin(position * div_term)  # crash ou résultat faux

# BUG 3 : oublier le unsqueeze(0) avant register_buffer
self.register_buffer("pe", pe)  # shape (max_len, d_model) au lieu de (1, max_len, d_model)
# → pe[:, :T] sélectionne les mauvaises dimensions
```

**Correction.** Vérifier que `pe[0, :4]` vaut `[sin(0), cos(0), sin(0), cos(0)] = [0, 1, 0, 1]` et que `pe[1, :4]` est différent de `pe[0, :4]`.

---
# Fin

On a couvert tout le spectre : les maths de base, la théorie complète du Transformer avec preuves, l'implémentation from scratch, un sanity check, des exercices, et les pièges à éviter.

Référence : Vaswani et al., *Attention Is All You Need*, NIPS 2017 (arXiv:1706.03762).